# 25 — LLM answer-quality pipeline comparison preflight

이 노트북은 **Codex coder agent**가 작성·실행한 개발셋 전용 offline/local preflight입니다. OLD-K3/K5와 STRUCT-K3/K5 네 end-to-end 패키지의 160개 단일 실행 payload를 결과 전에 동결합니다. 이번 단계에서는 OpenAI Responses API를 호출하지 않으며, holdout·Chroma·새 embedding·네트워크에 접근하지 않습니다.

K는 모든 cohort에서 **입력 unique-card evidence depth의 최대치**입니다. direct 답변의 카드 수는 가변이고 recommendation 답변은 최대 K입니다. 비교 결론은 청킹 단독이 아니라 검색·라우팅·reranker·evidence packaging을 포함한 OLD/STRUCT end-to-end 패키지에 한정합니다. 같은 schema·필드 순서·카드당 evidence cap·결정적 head truncation을 고정하지만 실제 context 길이는 조건별로 다를 수 있습니다. token-matched 진단은 이번 범위가 아닙니다.

## 사전 고정 계약

- 40 queries: proper 10, numeric 10, semantic 10, AND recommendation 10.
- 4 configurations × 40 = 160 payloads, one response each; repeated 320-run evaluation is excluded.
- OLD: D20 eligible leaf candidates, frozen query-text router, semantic only BGE augmented rerank.
- STRUCT: structural direct-body candidates, frozen automatic same-card one-hop bundle, all-query BGE. K3 uses D20 and K5 uses D30.
- Generation: exact `gpt-5.6-terra`, reasoning `medium`, tools 0, `store=false`, strict JSON schema, `max_output_tokens=1200`, SDK retries 0. Schema/refusal/incomplete failures are retained as failures.
- Report cohorts separately and compute direct/recommendation 50:50 family macro; overall micro is supplemental.
- Prompt/request input excludes pipeline label, K, cohort/category, gold cardinality, scores, paths/spans, labels, gates and evaluation fields.
- External execution remains fail-closed until exact approval-core authorization.

In [ ]:
# Offline response-level validator and reproducible notebook-content hash finalizer.
gold_by_payload={r['payload_id']:r for r in jl(OUT/'answer_gold.jsonl')}
audit_by_payload={r['payload_id']:r for r in csv.DictReader((OUT/'answer_gold_audit.csv').open(encoding='utf-8',newline=''))}
context_by_payload={r['payload_id']:r for r in jl(OUT/'contexts.jsonl')}
assert len(gold_by_payload)==len(audit_by_payload)==len(context_by_payload)==160
def code_cell_source_sha(doc,cell_id):
    cell=next(c for c in doc['cells'] if c.get('id')==cell_id); source=''.join(cell.get('source',[])) if isinstance(cell.get('source',[]),list) else cell.get('source',''); return hashlib.sha256(canon({'cell_id':cell_id,'source_text':source}).encode()).hexdigest()
validator_code_sha=code_cell_source_sha(_notebook_doc,'25-response-validator-source'); execution_code_sha=code_cell_source_sha(_notebook_doc,'25-external-fail-closed')
execution_cell=next(c for c in _notebook_doc['cells'] if c.get('id')=='25-external-fail-closed'); execution_source=''.join(execution_cell.get('source',[])) if isinstance(execution_cell.get('source',[]),list) else execution_cell.get('source','')
if re.search(r'(^|\n)\s*assert\b',execution_source): raise RuntimeError('External runner approval path must not use assert')
guard_position=execution_source.index("_require_external(os.environ.get('RUN_APPROVED_25_EXTERNAL')"); import_position=execution_source.index('from openai import OpenAI'); client_position=execution_source.index('client=OpenAI')
if not guard_position<import_position<client_position: raise RuntimeError('External guard must precede OpenAI import and client creation')
optimized_env=dict(os.environ); optimized_env['RUN_APPROVED_25_EXTERNAL']='0'; optimized_env['PYTHONOPTIMIZE']='1'; optimized_probe=subprocess.run([os.sys.executable,'-O','-c',execution_source],env=optimized_env,text=True,capture_output=True,timeout=30)
if optimized_probe.returncode==0 or 'External Responses execution is not approved' not in optimized_probe.stderr: raise RuntimeError('python -O external guard probe did not fail closed before import')
optimized_guard_audit={'python_optimize':1,'runner_assert_statements':0,'guard_before_openai_import':True,'guard_before_client_creation':True,'returncode_nonzero':True,'expected_runtime_error_observed':True,'openai_import_reached':False,'api_network_gpu_model':0,'execution_code_sha256':execution_code_sha}; write_json('external_runner_optimized_guard_audit.json',optimized_guard_audit)
abstain_expected={pid for pid,r in audit_by_payload.items() if r['abstain_expected']=='True'}
assert len(abstain_expected)==11 and all(not gold_by_payload[pid]['allowed_evidence_ids'] for pid in abstain_expected)
evidence_ids_by_payload={pid:{e['evidence_id'] for g in row['sent_groups'] for e in g['evidences']} for pid,row in context_by_payload.items()}
ownership_by_payload={}; evidence_text_by_payload={}
for pid,row in context_by_payload.items():
    owners={}; texts={}
    for group in row['sent_groups']:
        key=(group['issuer'],group['card_name']); assert key not in owners; ids={e['evidence_id'] for e in group['evidences']}; assert ids and not set(texts).intersection(ids); owners[key]=ids; texts.update({e['evidence_id']:e['text'] for e in group['evidences']})
    assert set(texts)==evidence_ids_by_payload[pid]; ownership_by_payload[pid]=owners; evidence_text_by_payload[pid]=texts
VALIDATION_REQUIRED_KEYS={'answer_type','summary','cards','claims','citations','insufficient_evidence'}
def validate_response(payload_id,response):
    if payload_id not in gold_by_payload: return {'payload_id':payload_id,'status':'format_failure','format_errors':['unknown_payload_id'],'quality_failures':[]}
    fmt=[]
    if not isinstance(response,dict): return {'payload_id':payload_id,'status':'format_failure','format_errors':['response_not_object'],'quality_failures':[]}
    if set(response)!=VALIDATION_REQUIRED_KEYS: fmt.append('top_level_keys_not_exact')
    if response.get('answer_type') not in {'direct_answer','card_recommendation','insufficient'}: fmt.append('invalid_answer_type')
    if not isinstance(response.get('summary'),str): fmt.append('summary_not_string')
    for key in ('cards','claims','citations'):
        if not isinstance(response.get(key),list): fmt.append(f'{key}_not_array')
    if not isinstance(response.get('insufficient_evidence'),bool): fmt.append('insufficient_evidence_not_boolean')
    if fmt: return {'payload_id':payload_id,'status':'format_failure','format_errors':fmt,'quality_failures':[]}
    cards=response['cards']; claims=response['claims']; citations=response['citations']
    if any(not isinstance(x,dict) or set(x)!={'issuer','card_name','summary','citations'} for x in cards): fmt.append('card_shape_invalid')
    if any(not isinstance(x,dict) or set(x)!={'card_name','claim','citations'} for x in claims): fmt.append('claim_shape_invalid')
    if any(not isinstance(x,dict) or set(x)!={'evidence_id','supports'} for x in citations): fmt.append('citation_shape_invalid')
    if fmt: return {'payload_id':payload_id,'status':'format_failure','format_errors':fmt,'quality_failures':[]}
    quality=[]; actual=evidence_ids_by_payload[payload_id]; owners=ownership_by_payload[payload_id]; evidence_text=evidence_text_by_payload[payload_id]; gold=gold_by_payload[payload_id]; expected_abstain=payload_id in abstain_expected
    card_refs=[]; claim_refs=[]
    for card in cards:
        if not isinstance(card['issuer'],str) or not card['issuer'].strip() or not isinstance(card['card_name'],str) or not card['card_name'].strip() or not isinstance(card['summary'],str) or not card['summary'].strip(): quality.append('card_required_text_empty')
        if not isinstance(card['citations'],list) or not card['citations'] or any(not isinstance(x,str) or not x for x in card['citations']): quality.append('card_citations_empty_or_invalid')
        else:
            card_refs.extend(card['citations']); owner_ids=owners.get((card['issuer'],card['card_name']))
            if owner_ids is None: quality.append('card_identity_not_in_transmitted_group')
            elif not set(card['citations']).issubset(owner_ids): quality.append('card_citation_crosses_group_ownership')
    card_pairs={(x['issuer'],x['card_name']) for x in cards if isinstance(x.get('issuer'),str) and isinstance(x.get('card_name'),str)}
    if len(card_pairs)!=len(cards): quality.append('duplicate_card_identity')
    for claim in claims:
        matching=[card for card in cards if isinstance(claim['card_name'],str) and card.get('card_name')==claim['card_name']]
        if len(matching)!=1: quality.append('claim_card_identity_not_unique_in_returned_cards')
        if not isinstance(claim['claim'],str) or not claim['claim'].strip(): quality.append('claim_text_empty')
        if not isinstance(claim['citations'],list) or not claim['citations'] or any(not isinstance(x,str) or not x for x in claim['citations']): quality.append('claim_citations_empty_or_invalid')
        else:
            claim_refs.extend(claim['citations'])
            if len(matching)==1:
                owner_ids=owners.get((matching[0]['issuer'],matching[0]['card_name']))
                if owner_ids is None or not set(claim['citations']).issubset(owner_ids): quality.append('claim_citation_crosses_card_group_ownership')
            cited_text=' '.join(evidence_text[eid] for eid in claim['citations'] if eid in evidence_text); claim_terms=set(re.findall(r'[0-9a-z가-힣]{2,}',norm(claim['claim']))); cited_terms=set(re.findall(r'[0-9a-z가-힣]{2,}',norm(cited_text))); required_overlap=min(2,len(claim_terms))
            if required_overlap and len(claim_terms & cited_terms)<required_overlap: quality.append('claim_has_no_minimum_lexical_grounding_in_owned_evidence')
    top_refs=[]
    for citation in citations:
        if not isinstance(citation['evidence_id'],str) or not citation['evidence_id'] or not isinstance(citation['supports'],str) or not citation['supports'].strip(): quality.append('top_level_citation_empty')
        else: top_refs.append(citation['evidence_id'])
    all_refs=set(card_refs+claim_refs+top_refs)
    if not all_refs.issubset(actual): quality.append('citation_not_in_payload_evidence')
    if len(top_refs)!=len(set(top_refs)): quality.append('duplicate_top_level_citation')
    if set(top_refs)!=set(card_refs+claim_refs): quality.append('top_level_citations_not_exact_reference_union')
    is_insufficient=response['answer_type']=='insufficient'
    if is_insufficient:
        if not response['insufficient_evidence'] or cards or claims or citations: quality.append('insufficient_state_inconsistent')
    else:
        if response['insufficient_evidence'] or not cards or not claims or not citations: quality.append('answer_state_missing_minimum_content')
        expected_type='direct_answer' if gold['task']=='direct' else 'card_recommendation'
        if response['answer_type']!=expected_type: quality.append('answer_type_task_mismatch')
    if expected_abstain and not is_insufficient: quality.append('expected_abstention_not_returned')
    if not expected_abstain and is_insufficient: quality.append('unexpected_abstention_with_sufficient_gold_evidence')
    return {'payload_id':payload_id,'status':'semantic_validation_failure' if quality else 'transport_semantic_validation_pass','format_errors':[],'quality_failures':sorted(set(quality)),'abstain_expected':expected_abstain,'factual_correctness_scored':False}
validation_contract={'version':'25-response-validator-v3','validator_code_sha256':validator_code_sha,'execution_code_sha256':execution_code_sha,'optimized_guard_audit_sha256':sha(OUT/'external_runner_optimized_guard_audit.json'),'python_optimize_guard_verified':True,'pre_api_fail_closed':['approval env by explicit RuntimeError guard','approval core canonical SHA','payload/prompt/schema hashes','answer gold freeze hashes','validator contract SHA','validator cell source SHA','external runner cell source SHA','optimized guard audit SHA','execution guard SHA'],'response_storage_order':'raw response first, validation immediately afterward','batch_behavior':'format and semantic validation failures are retained per response and do not abort the remaining single-run batch','pass_status':'transport_semantic_validation_pass','pass_scope':'transport shape, answer-state consistency, evidence existence, card-group ownership, and minimum lexical grounding only; not factual correctness','format_failure_definition':['non-object or non-exact top-level keys','invalid primitive/container type','invalid card/claim/citation shape','provider refusal/incomplete/no JSON'],'semantic_validation_failure_definition':['minimum content missing when evidence is sufficient','citation not present in transmitted evidence','issuer/card identity absent from transmitted groups','card or claim citation crosses group ownership','claim lacks minimum lexical grounding','top-level citation set inconsistent','answer/insufficient/task state inconsistent','expected abstention mismatch'],'expected_abstention_rows':11,'expected_abstention_source':'answer_gold_audit.csv abstain_expected=true','minimum_sufficient_answer':{'cards':1,'claims':1,'citations':1},'citation_contract':'every returned issuer/card is an exact transmitted group; card and claim citations stay within that group; top-level citations equal their union','whole_batch_abort_on_response_failure':False,'factual_correctness_scoring':'required after external responses; answer-gold facts/conditions/units determine correctness','required_post_execution_output':'answer_quality_scores.jsonl','external_run_complete_only_after_factual_scoring':True,'external_api_requests_current':0}
write_json('response_validation_contract.json',validation_contract)
non_abstain=next(pid for pid in gold_by_payload if pid not in abstain_expected); ctx=context_by_payload[non_abstain]; eid=next(iter(gold_by_payload[non_abstain]['allowed_evidence_ids'])); hit=next((g,e) for g in ctx['sent_groups'] for e in g['evidences'] if e['evidence_id']==eid); group,evidence=hit; claim_text=next(line.strip() for line in evidence['text'].splitlines() if len(re.findall(r'[0-9a-z가-힣]{2,}',norm(line)))>=2)
good={'answer_type':'direct_answer' if gold_by_payload[non_abstain]['task']=='direct' else 'card_recommendation','summary':'근거로 확인됨','cards':[{'issuer':group['issuer'],'card_name':group['card_name'],'summary':'근거 요약','citations':[eid]}],'claims':[{'card_name':group['card_name'],'claim':claim_text,'citations':[eid]}],'citations':[{'evidence_id':eid,'supports':'주장을 지원'}],'insufficient_evidence':False}
abstain_pid=sorted(abstain_expected)[0]; abstain={'answer_type':'insufficient','summary':'근거 부족','cards':[],'claims':[],'citations':[],'insufficient_evidence':True}
unknown=json.loads(json.dumps(good)); unknown['cards'][0]['citations']=['E999']; unknown['claims'][0]['citations']=['E999']; unknown['citations'][0]['evidence_id']='E999'
wrong_identity=json.loads(json.dumps(good)); wrong_identity['cards'][0]['issuer']='존재하지않는발급사'
other=next(g for g in ctx['sent_groups'] if (g['issuer'],g['card_name'])!=(group['issuer'],group['card_name'])); other_eid=other['evidences'][0]['evidence_id']; cross_card=json.loads(json.dumps(good)); cross_card['cards'][0]['citations']=[other_eid]; cross_card['claims'][0]['citations']=[other_eid]; cross_card['citations']=[{'evidence_id':other_eid,'supports':'잘못 연결'}]
arbitrary=json.loads(json.dumps(good)); arbitrary['claims'][0]['claim']='달 표면 양자컴퓨터 영구 대여'
synthetic=[{'case':'sufficient_transport_semantic_pass','result':validate_response(non_abstain,good)},{'case':'expected_abstention_transport_semantic_pass','result':validate_response(abstain_pid,abstain)},{'case':'unknown_citation_semantic_failure','result':validate_response(non_abstain,unknown)},{'case':'missing_keys_format_failure','result':validate_response(non_abstain,{'answer_type':'direct_answer'})},{'case':'wrong_issuer_card_semantic_failure','result':validate_response(non_abstain,wrong_identity)},{'case':'cross_card_citation_semantic_failure','result':validate_response(non_abstain,cross_card)},{'case':'arbitrary_claim_minimum_grounding_failure','result':validate_response(non_abstain,arbitrary)}]
expected_status=['transport_semantic_validation_pass','transport_semantic_validation_pass','semantic_validation_failure','format_failure','semantic_validation_failure','semantic_validation_failure','semantic_validation_failure']; assert [x['result']['status'] for x in synthetic]==expected_status; write_json('response_validation_synthetic_audit.json',{'cases':synthetic,'expected_status':expected_status,'pass':True,'factual_correctness_not_scored':True})
approval_doc=json.loads((OUT/'approval_core.json').read_text()); approval_core=approval_doc['approval_core']; approval_core.update({'response_validation_contract_sha256':sha(OUT/'response_validation_contract.json'),'validator_code_sha256':validator_code_sha,'execution_code_sha256':execution_code_sha,'optimized_guard_audit_sha256':sha(OUT/'external_runner_optimized_guard_audit.json'),'required_post_execution_output':'answer_quality_scores.jsonl'}); approval_sha=hashlib.sha256(canon(approval_core).encode()).hexdigest(); write_json('approval_core.json',{'approval_core':approval_core,'approval_core_sha256':approval_sha})
gold_freeze=json.loads((OUT/'answer_gold_freeze.json').read_text()); guard_core={'approval_core_sha256':approval_sha,'payloads_sha256':sha(OUT/'payloads.jsonl'),'prompt_sha256':sha(OUT/'prompt.json'),'schema_sha256':sha(OUT/'response_schema.json'),'answer_gold_sha256':sha(OUT/'answer_gold.jsonl'),'answer_gold_freeze_sha256':sha(OUT/'answer_gold_freeze.json'),'response_validation_contract_sha256':sha(OUT/'response_validation_contract.json'),'validator_code_sha256':validator_code_sha,'execution_code_sha256':execution_code_sha,'optimized_guard_audit_sha256':sha(OUT/'external_runner_optimized_guard_audit.json'),'response_count':160,'sdk_retries':0,'required_post_execution_output':'answer_quality_scores.jsonl','external_run_complete_only_after_factual_scoring':True,'required_approval_env':'APPROVED_25_EXECUTION_GUARD_SHA256'}; guard_sha=hashlib.sha256(canon(guard_core).encode()).hexdigest(); write_json('external_execution_guard.json',{'guard_core':guard_core,'execution_guard_sha256':guard_sha})
status=json.loads((OUT/'approval_status.json').read_text()); status.update({'approval_core_sha256':approval_sha,'execution_guard_sha256':guard_sha,'validator_code_sha256':validator_code_sha,'execution_code_sha256':execution_code_sha,'external_execution_allowed':False,'external_requests_current':0,'response_validator_ready':True,'factual_quality_scorer_required_after_external':True}); write_json('approval_status.json',status)
payload_manifest=json.loads((OUT/'payload_manifest.json').read_text()); payload_manifest['response_validation_contract_sha256']=sha(OUT/'response_validation_contract.json'); payload_manifest['approval_core_sha256']=approval_sha; payload_manifest['execution_guard_sha256']=guard_sha; write_json('payload_manifest.json',payload_manifest)
def notebook_content_sha(path):
    doc=json.loads(path.read_text()); stable={'nbformat':doc['nbformat'],'nbformat_minor':doc['nbformat_minor'],'cells':[{'cell_type':c['cell_type'],'id':c.get('id'),'source_text':''.join(c.get('source',[])) if isinstance(c.get('source',[]),list) else c.get('source',''),'tags':c.get('metadata',{}).get('tags',[])} for c in doc['cells']]}; return hashlib.sha256(canon(stable).encode()).hexdigest()
notebook_content_hash=notebook_content_sha(NOTEBOOK)
readme=(OUT/'README.md').read_text(); readme+='''
응답 저장 직후 validator는 형식 실패와 transport-semantic 실패를 분리합니다. 실제 전송 card group의 발급사·카드명·evidence 소유권, citation 일관성, 최소 lexical grounding과 abstention 상태를 검사합니다. 통과 상태 `transport_semantic_validation_pass`는 사실 정답 판정이 아닙니다. 사실·수치·조건 정답은 외부 응답 뒤 `answer_quality_scores.jsonl`을 만드는 별도 gold quality scorer가 반드시 판정해야 하며, 그 전에는 외부 run을 완료로 보지 않습니다. 개별 실패는 저장·채점하며 160개 batch 전체를 중단하지 않습니다. `abstain_expected=true` 11건은 예상 abstention 계약에 결속했습니다.

노트북 무결성은 raw 파일의 자기 hash를 내부에 넣지 않습니다. cell source, cell id, cell type, skip tag와 nbformat만 canonical JSON으로 해시하고 outputs, execution_count와 실행 timing metadata는 제외합니다. 따라서 같은 소스의 fresh 실행에서 같은 `notebook_content_sha256`이 재현됩니다.
External runner의 승인·환경·hash 검사는 `assert`가 아니라 명시적 RuntimeError guard를 사용합니다. Runner source 자체를 `python -O`와 승인 env 0으로 실행해 OpenAI import/client 생성 전에 차단되는지 검증하고 audit SHA를 approval core에 포함합니다.
''' ; (OUT/'README.md').write_text(readme,encoding='utf-8')
extra=['response_validation_contract.json','response_validation_synthetic_audit.json','external_execution_guard.json','external_runner_optimized_guard_audit.json']; manifest=json.loads((OUT/'run_manifest.json').read_text()); output_names=sorted(set(manifest['outputs'])|set(extra)); manifest['outputs']={n:sha(OUT/n) for n in output_names}; manifest['notebook_content_sha256']=notebook_content_hash; manifest['notebook_content_hash_policy']='canonical cell type/id/source text/skip tags + nbformat; source list/string representation normalized; excludes outputs, execution_count, timing metadata, and manifest values'; manifest['self_hash_policy']='run_manifest.json and integrity.json excluded'; write_json('run_manifest.json',manifest)
integrity=json.loads((OUT/'integrity.json').read_text()); integrity['output_hashes']=manifest['outputs']; integrity['notebook_content_sha256']=notebook_content_hash; integrity['notebook_content_hash_policy']=manifest['notebook_content_hash_policy']; integrity['response_validation_ready']=True; integrity['validator_code_sha256']=validator_code_sha; integrity['execution_code_sha256']=execution_code_sha; integrity['optimized_guard_audit_sha256']=sha(OUT/'external_runner_optimized_guard_audit.json'); integrity['python_optimize_external_guard_pass']=True; integrity['expected_abstention_rows']=11; integrity['synthetic_validator_cases']=7; integrity['factual_quality_scorer_required_after_external']=True; integrity['placeholder_count_final']=0; integrity['run_manifest_sha256']=sha(OUT/'run_manifest.json'); write_json('integrity.json',integrity)
assert manifest['notebook_content_sha256']==integrity['notebook_content_sha256'] and integrity['placeholder_count_final']==0 and payload_manifest['external_requests_current']==0
print({'response_validator':'PASS','expected_abstention_rows':11,'synthetic_cases':7,'validator_code_sha256':validator_code_sha,'execution_code_sha256':execution_code_sha,'approval_core_sha256':approval_sha,'execution_guard_sha256':guard_sha,'notebook_content_sha256':notebook_content_hash,'external_requests':0})


In [1]:
from pathlib import Path
from collections import defaultdict
import csv, hashlib, json, math, os, re, resource, statistics, subprocess, time, unicodedata
import numpy as np
import tiktoken
cwd=Path.cwd().resolve(); ROOT=cwd if (cwd/'notebooks').is_dir() else (cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None); assert ROOT is not None
assert os.environ.get('RUN_APPROVED_25_EXTERNAL','0')=='0' and os.environ.get('HF_HUB_OFFLINE')=='1' and os.environ.get('TRANSFORMERS_OFFLINE')=='1'
OUT=ROOT/'notebooks/data/25_llm_answer_quality_pipeline_comparison'; OUT.mkdir(parents=True,exist_ok=True)
P13=ROOT/'notebooks/data/13_hierarchical_chunking_retrieval'; P16=ROOT/'notebooks/data/16_normalized_rrf_weight_ablation'; P21=ROOT/'notebooks/data/21_current_chunking_prebranch_reranker_evaluation'; P22=ROOT/'notebooks/data/22_structural_heading_chunking_ablation'; P24=ROOT/'notebooks/data/24_multicard_recommendation_retrieval_evaluation'; BGE=ROOT/'.cache/reranker/bge-reranker-v2-m3'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest(); canon=lambda x:json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':')); norm=lambda x:' '.join(unicodedata.normalize('NFKC',str(x)).lower().split())
def jl(path): return [json.loads(x) for x in Path(path).read_text(encoding='utf-8').splitlines() if x]
def write_json(name,value): (OUT/name).write_text(json.dumps(value,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
def write_jsonl(name,rows): (OUT/name).write_text(''.join(canon(r)+'\n' for r in rows),encoding='utf-8')
def write_csv(name,rows):
    with (OUT/name).open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
def tree(root):
    files={str(p.relative_to(root)):sha(p) for p in sorted(root.rglob('*')) if p.is_file()}
    return {'file_count':len(files),'total_bytes':sum((root/n).stat().st_size for n in files),'digest':hashlib.sha256(canon(files).encode()).hexdigest(),'files':files}
INPUTS=[P13/'retrieval_per_query.csv',P13/'chunks.jsonl',P16/'rrf_weight_candidates.csv',P21/'phase_a_classification.csv',P21/'leaf_pair_scores.csv',P22/'followup2_rankings.csv',P22/'chunks.jsonl',P22/'hierarchy_manifest.jsonl',P24/'followup1_contract.json',P24/'followup2_queries.csv',P24/'followup2_query_classification.csv',P24/'followup2_scored_rankings.jsonl',P24/'followup2_bundles.jsonl',P24/'followup2_pair_scores.csv',P24/'followup2_gold_labels.csv',P24/'followup2_atomic_claim_audit.jsonl',P24/'gold_card_labels.csv',P24/'followup5_bundles.jsonl',P24/'followup5_rankings.jsonl',P24/'followup5_pair_scores.csv']
input_before={str(p.relative_to(ROOT)):sha(p) for p in INPUTS}; bge_before=tree(BGE)
SYSTEM_PROMPT='''너는 제공된 카드 근거만 사용해 질문에 답하는 도우미다. 근거에 없는 사실은 만들지 말고, 부족하면 insufficient_evidence를 true로 표시한다. 카드명·발급사·혜택·숫자·조건에 관한 모든 주장은 실제로 존재하는 evidence_id를 인용한다. 수치, 단위, 실적, 한도, 횟수, 제외 조건을 확대하거나 축약하지 않는다. 발급 가능성, 시장 전체에서의 최적성, 개인 자격 충족 여부를 주장하지 않는다. 근거가 충돌하면 summary와 claim에 충돌을 명시한다. 여러 카드가 조건을 만족하는 질문에서는 근거 순서를 유지하고, 질문의 모든 필수 조건을 근거로 확인한 카드만 포함한다. 지정된 JSON schema만 반환한다.'''.strip()
USER_TEMPLATE='''질문:\n{query}\n\n근거 카드 그룹(JSON):\n{evidence_json}\n\n근거에 의해 직접 뒷받침되는 내용만 구조화해 답하라.'''.strip()
SCHEMA={'type':'object','additionalProperties':False,'required':['answer_type','summary','cards','claims','citations','insufficient_evidence'],'properties':{'answer_type':{'type':'string','enum':['direct_answer','card_recommendation','insufficient']},'summary':{'type':'string'},'cards':{'type':'array','maxItems':5,'items':{'type':'object','additionalProperties':False,'required':['issuer','card_name','summary','citations'],'properties':{'issuer':{'type':'string'},'card_name':{'type':'string'},'summary':{'type':'string'},'citations':{'type':'array','items':{'type':'string'}}}}},'claims':{'type':'array','items':{'type':'object','additionalProperties':False,'required':['card_name','claim','citations'],'properties':{'card_name':{'type':'string'},'claim':{'type':'string'},'citations':{'type':'array','minItems':1,'items':{'type':'string'}}}}},'citations':{'type':'array','items':{'type':'object','additionalProperties':False,'required':['evidence_id','supports'],'properties':{'evidence_id':{'type':'string'},'supports':{'type':'string'}}}},'insufficient_evidence':{'type':'boolean'}}}
SCHEMAS={}
for k in (3,5):
    schema=json.loads(json.dumps(SCHEMA)); schema['properties']['cards']['maxItems']=k; SCHEMAS[f'K{k}']=schema
CONFIGS={'OLD-K3':{'pipeline':'OLD','input_unique_card_depth':3,'candidate_depth':20},'OLD-K5':{'pipeline':'OLD','input_unique_card_depth':5,'candidate_depth':20},'STRUCT-K3':{'pipeline':'STRUCT','input_unique_card_depth':3,'candidate_depth':20},'STRUCT-K5':{'pipeline':'STRUCT','input_unique_card_depth':5,'candidate_depth':30}}
CONTRACT={'name':'25 LLM answer-quality 4-package preflight','provenance':'Codex coder agent','declared_before_answer_results':True,'scope':'fixed 10-card development corpus; 40 queries; 160 payload single run','comparison_unit':'OLD versus STRUCT end-to-end packages, not chunking-only causality','configurations':CONFIGS,'k_definition':'maximum input unique-card evidence groups for every cohort','output_card_count':{'direct':'variable; only evidenced cards','recommendation':'at most input K'},'context_fairness':{'same_schema_and_field_order':True,'max_evidence_per_card':5,'per_evidence_cl100k_head_cap':640,'actual_context_length_may_differ':True,'token_matched_diagnostic':False},'generation':{'api':'OpenAI Responses API','model':'gpt-5.6-terra','reasoning_effort':'medium','tools':[],'store':False,'strict_structured_output':True,'max_output_tokens':1200,'sdk_retries':0,'single_run_responses':160,'repeat_run_excluded':True,'schema_refusal_incomplete_are_failures':True},'transmitted_allowlist':['query text','anonymous evidence_id','issuer','card_name','heading','evidence text'],'transmitted_forbidden':['OLD','STRUCT','K','cohort','category','gold cardinality','retrieval/reranker score','path/span','label','gate','evaluation fields'],'reporting':{'cohorts':['proper_noun','numeric_condition','semantic','and_recommendation'],'family_macro':'direct and recommendation weighted 50:50','overall_micro':'supplemental'},'gold_requirements':['task','stable relevant card IDs and variants','allowed facts','units','conditions','allowed evidence IDs','abstain rule','recommendation set-only target'],'price_contract':{'model':'gpt-5.6-terra','tier':'short-context standard','input_usd_per_million_tokens':2.0,'output_usd_per_million_tokens':12.0,'source_url':'https://openai.com/api/pricing/','checked_date':'2026-08-27','verification':'user-provided contract recorded offline; URL not fetched in this run'},'execution':{'external_api_requests':0,'network':0,'new_embedding':0,'chroma':0,'holdout':0,'package_install':0}}
CONTRACT['output_card_count']['strict_schema_card_max_matches_input_k']=True
write_json('evaluation_contract.json',CONTRACT); write_json('prompt.json',{'system_prompt':SYSTEM_PROMPT,'user_template':USER_TEMPLATE,'forbidden_internal_fields':CONTRACT['transmitted_forbidden']}); write_json('response_schema.json',SCHEMAS)
write_json('input_manifest.json',{'inputs_before':input_before,'bge_before':bge_before})
print({'contract':'FROZEN','inputs':len(INPUTS),'bge_files':bge_before['file_count'],'external_requests':0})


{'contract': 'FROZEN', 'inputs': 20, 'bge_files': 40, 'external_requests': 0}


In [2]:
# Build all candidate/bundle inputs without reading answer gold fields.
direct_source=list(csv.DictReader((P13/'retrieval_per_query.csv').open(encoding='utf-8',newline=''))); direct_q={}
for r in direct_source:
    slim=(r['query'],r['category'])
    if r['query_id'] in direct_q: assert direct_q[r['query_id']]==slim
    else: direct_q[r['query_id']]=slim
assert len(direct_q)==30 and {c:sum(v[1]==c for v in direct_q.values()) for c in ['proper_noun','numeric_condition','semantic']}=={'proper_noun':10,'numeric_condition':10,'semantic':10}
cmb_rows=[r for r in csv.DictReader((P24/'followup2_queries.csv').open(encoding='utf-8',newline='')) if r['cohort']=='and_combination']; assert len(cmb_rows)==10
queries=[]
for qid,(text,cat) in sorted(direct_q.items()): queries.append({'query_id':qid,'query_text':text,'cohort':cat,'task':'direct'})
for r in sorted(cmb_rows,key=lambda x:x['query_id']): queries.append({'query_id':r['query_id'],'query_text':r['query_text'],'cohort':'and_recommendation','task':'recommendation'})
assert len(queries)==40 and len({r['query_id'] for r in queries})==40; query_text={r['query_id']:r['query_text'] for r in queries}; write_csv('queries.csv',[{'request_order':i+1,**r,'query_sha256':hashlib.sha256(r['query_text'].encode()).hexdigest()} for i,r in enumerate(queries)])
old_chunks=jl(P13/'chunks.jsonl'); old_by={r['id']:r for r in old_chunks}; assert len(old_by)==327
struct_chunks=jl(P22/'chunks.jsonl'); struct_by={r['chunk_id']:r for r in struct_chunks}; hierarchy=jl(P22/'hierarchy_manifest.jsonl'); node_by={r['node_id']:r for r in hierarchy}; children=defaultdict(list)
for n in hierarchy:
    if n['parent_id'] is not None: children[n['parent_id']].append(n)
for rows in children.values(): rows.sort(key=lambda n:((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
route={r['query_id']:r for r in csv.DictReader((P21/'phase_a_classification.csv').open(encoding='utf-8',newline=''))}; assert len(route)==30 and all(route[q]['predicted_category']==direct_q[q][1] for q in direct_q)
cand=defaultdict(list)
for r in csv.DictReader((P16/'rrf_weight_candidates.csv').open(encoding='utf-8',newline='')):
    if r['configuration']=='vector_0.4_bm25_0.6' and r['query_id'] in direct_q and r['level'] in {'section','benefit'}: cand[r['query_id']].append((int(r['fused_rank']),r['chunk_id']))
bge_old={(r['query_id'],r['chunk_id']):float(r['reranker_logit']) for r in csv.DictReader((P21/'leaf_pair_scores.csv').open(encoding='utf-8',newline='')) if r['model']=='bge' and r['input_variant']=='augmented'}
old_ranked={}
for q in direct_q:
    base=sorted(cand[q])[:20]; assert len(base)==20
    if route[q]['predicted_category']=='semantic': ranked=[cid for rank,cid in sorted(base,key=lambda x:(-bge_old[(q,x[1])],x[0],x[1]))]
    else: ranked=[cid for rank,cid in base]
    old_ranked[q]=ranked
f2_scored={r['query_id']:r for r in jl(P24/'followup2_scored_rankings.jsonl') if r['cohort']=='and_combination'}; assert len(f2_scored)==10
for q in f2_scored: old_ranked[q]=f2_scored[q]['old_chunk_ids']; assert len(old_ranked[q])==20
rank22=[]
for r in csv.DictReader((P22/'followup2_rankings.csv').open(encoding='utf-8',newline='')):
    if abs(float(r['k1'])-1.5)<1e-12 and abs(float(r['b'])-.75)<1e-12 and abs(float(r['vector_weight'])-.4)<1e-12 and abs(float(r['bm25_weight'])-.6)<1e-12: rank22.append(r)
assert len(rank22)==30 and {r['query_id'] for r in rank22}==set(direct_q)
bundle_rows=[]; trace_rows=[]
for r in rank22:
    full=json.loads(r['fused_top50_chunk_ids'])
    for depth in (20,30):
        top=full[:depth]; by_card=defaultdict(list)
        for rank,cid in enumerate(top,1): by_card[struct_by[cid]['metadata']['card_key']].append((rank,cid))
        for card,seeds in sorted(by_card.items(),key=lambda x:(x[1][0][0],x[0])):
            best_rank,seed_id=seeds[0]; seed=struct_by[seed_id]; node=node_by[seed['metadata']['node_id']]; selected=[]; relations=[]; parent_context=None
            def add(cid,kind,source_node):
                if len(selected)>=5 or cid in selected: return
                c=struct_by[cid]; assert c['metadata']['card_key']==card
                selected.append(cid); relations.append({'selection_order':len(selected),'chunk_id':cid,'relation':kind,'source_relation_node_id':source_node})
            add(seed_id,'best_rrf_seed',node['node_id'])
            same=[struct_by[c] for c in node['search_chunk_ids'] if c!=seed_id]; same.sort(key=lambda c:(abs(c['metadata']['part_index']-seed['metadata']['part_index']),c['metadata']['part_index'],c['chunk_id']))
            for c in same: add(c['chunk_id'],'same_node_adjacent_part',node['node_id'])
            parent=node_by.get(node['parent_id'])
            if parent is not None and parent['parent_id'] is not None:
                if not parent['heading_only']:
                    for cid in parent['search_chunk_ids']: add(cid,'non_root_immediate_parent_direct_body',parent['node_id'])
                else:
                    parent_context=parent['heading_text']; seed_line=node['heading_line_number'] if node['heading_line_number'] is not None else 10**12
                    siblings=[n for n in children[parent['node_id']] if not n['heading_only'] and n['search_chunk_ids']]; siblings.sort(key=lambda n:(abs((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12)-seed_line),(n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
                    for n in siblings:
                        for cid in n['search_chunk_ids']: add(cid,'heading_only_parent_same_parent_direct_body_child',parent['node_id'])
            if node['parent_id'] is not None:
                for child in children[node['node_id']]:
                    if not child['heading_only']:
                        for cid in child['search_chunk_ids']: add(cid,'seed_node_direct_child',node['node_id'])
            for _,cid in seeds[1:]: add(cid,'same_card_other_top_depth_seed',node['node_id'])
            sections=['[카드]\n'+seed['metadata']['issuer']+' > '+seed['metadata']['card_name']]
            if parent_context: sections.append('[상위 제목]\n'+parent_context)
            for i,cid in enumerate(selected,1):
                c=struct_by[cid]; heading=' > '.join(c['heading_path']) if c['heading_path'] else '(root content)'; sections.append(f'[근거 {i} 경로]\n{heading}\n[근거 {i} 본문]\n{c["evidence_text"]}')
            text='\n\n'.join(sections); row={'query_id':r['query_id'],'depth':depth,'card_key':card,'best_seed_rrf_rank':best_rank,'best_seed_chunk_id':seed_id,'selected_chunk_ids':selected,'optional_parent_heading':parent_context,'bundle_text':text,'bundle_sha256':hashlib.sha256(text.encode()).hexdigest(),'relation_trace':relations,'root_seed':node['parent_id'] is None,'root_children_fanout':False}; bundle_rows.append(row)
            for item in relations: trace_rows.append({'query_id':r['query_id'],'depth':depth,'card_key':card,'root_seed':row['root_seed'],'root_children_fanout':False,**item})
assert all(1<=len(r['selected_chunk_ids'])<=5 and not r['root_children_fanout'] for r in bundle_rows) and all(not (r['root_seed'] and any(x['relation']=='seed_node_direct_child' for x in r['relation_trace'])) for r in bundle_rows)
write_jsonl('structural_direct_bundles.jsonl',bundle_rows); write_csv('structural_direct_bundle_trace.csv',trace_rows)
pair_plan={(r['query_id'],r['bundle_sha256']):r for r in bundle_rows}; assert all(q in direct_q for q,_ in pair_plan)
preflight={'status':'PASS_CPU_BUILD_BEFORE_GOLD','gold_fields_accessed':['none for ranking/bundle; retrieval_per_query allowlist only query_id/query/category'],'queries':40,'direct_queries':30,'recommendation_queries':10,'structural_direct_bundle_rows':len(bundle_rows),'structural_direct_unique_query_bundle_pairs':len(pair_plan),'old_candidate_depth':20,'structural_depths':[20,30],'root_children_fanout':0,'contract_sha256':sha(OUT/'evaluation_contract.json'),'bundles_sha256':sha(OUT/'structural_direct_bundles.jsonl'),'input_before':input_before,'bge_before':bge_before,'execution':{'external_api':0,'network':0,'new_embedding':0,'chroma':0}}; write_json('local_preflight.json',preflight)
assert {str(p.relative_to(ROOT)):sha(p) for p in INPUTS}==input_before and tree(BGE)==bge_before
print({'cpu_build':'PASS','queries':40,'bundle_rows':len(bundle_rows),'unique_bge_pairs':len(pair_plan),'gold_semantic_reads':0})


{'cpu_build': 'PASS', 'queries': 40, 'bundle_rows': 453, 'unique_bge_pairs': 327, 'gold_semantic_reads': 0}


In [ ]:
# Local BGE only. External API/network remain disabled.
assert os.environ.get('RUN_APPROVED_25_LOCAL_BGE')=='1' and os.environ.get('CUDA_VISIBLE_DEVICES')=='0' and os.environ.get('RUN_APPROVED_25_EXTERNAL','0')=='0'
import gc, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
def gpu_snapshot():
    gpu=subprocess.run(['nvidia-smi','--query-gpu=index,uuid,name,memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip().splitlines(); line=next(x for x in gpu if x.split(',')[0].strip()=='0'); uuid=line.split(',')[1].strip(); raw=subprocess.run(['nvidia-smi','--query-compute-apps=gpu_uuid,pid,process_name,used_gpu_memory','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip(); procs=[]
    for x in raw.splitlines():
        p=[v.strip() for v in x.split(',',3)]
        if len(p)==4 and p[0]==uuid: procs.append({'pid':int(p[1]),'process_name':p[2],'used_memory_mib':float(p[3])})
    return {'gpu_line':line,'processes':procs,'captured_at_unix':time.time()}
gpu_before=gpu_snapshot(); assert torch.cuda.is_available() and torch.cuda.device_count()==1 and '3090' in torch.cuda.get_device_name(0)
pairs=sorted(pair_plan.items()); tokenizer=AutoTokenizer.from_pretrained(BGE,local_files_only=True,trust_remote_code=False); audits=[]
for (qid,bsha),b in pairs:
    q=query_text[qid]; qn=len(tokenizer.encode(q,add_special_tokens=False)); dn=len(tokenizer.encode(b['bundle_text'],add_special_tokens=False)); enc=tokenizer(q,b['bundle_text'],truncation='only_second',max_length=8192); truncated=max(0,qn+dn+tokenizer.num_special_tokens_to_add(pair=True)-8192); assert dn<=4096 and truncated==0
    audits.append({'query_id':qid,'query_text_sha256':hashlib.sha256(q.encode()).hexdigest(),'bundle_sha256':bsha,'card_key':b['card_key'],'query_tokens':qn,'bundle_tokens':dn,'input_tokens':len(enc['input_ids']),'document_truncated_tokens':truncated,'gold_or_evaluation_fields_used':False})
load_start=time.perf_counter(); model=AutoModelForSequenceClassification.from_pretrained(BGE,local_files_only=True,trust_remote_code=False,dtype=torch.float16).eval().to('cuda:0'); load_seconds=time.perf_counter()-load_start; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
warm=tokenizer(['준비'],['준비'],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0'); t=time.perf_counter()
with torch.inference_mode(): assert torch.isfinite(model(**warm).logits).all()
torch.cuda.synchronize(); warmup_seconds=time.perf_counter()-t; del warm
scores={}; t=time.perf_counter()
for i in range(0,len(pairs),2):
    part=pairs[i:i+2]; batch=tokenizer([query_text[k[0]] for k,b in part],[b['bundle_text'] for k,b in part],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0')
    with torch.inference_mode(): logits=model(**batch).logits.reshape(-1).float().cpu().numpy()
    assert len(logits)==len(part) and np.isfinite(logits).all(); scores.update({k:float(v) for (k,b),v in zip(part,logits)}); del batch,logits
torch.cuda.synchronize(); scoring_seconds=time.perf_counter()-t; peak_alloc=torch.cuda.max_memory_allocated()/2**30; peak_reserved=torch.cuda.max_memory_reserved()/2**30; del model,tokenizer; gc.collect(); torch.cuda.empty_cache(); gpu_after=gpu_snapshot()
assert set(scores)==set(pair_plan) and all(math.isfinite(v) for v in scores.values())
score_rows=[{'query_id':qid,'query_text_sha256':hashlib.sha256(query_text[qid].encode()).hexdigest(),'bundle_sha256':bsha,'card_key':pair_plan[(qid,bsha)]['card_key'],'raw_logit':scores[(qid,bsha)]} for qid,bsha in sorted(scores)]; write_csv('local_bge_pair_scores.csv',score_rows); write_csv('local_bge_scorer_audit.csv',audits)
direct_struct_rank={}
for depth in (20,30):
    for q in direct_q:
        rows=[r for r in bundle_rows if r['query_id']==q and r['depth']==depth]; rows.sort(key=lambda r:(-scores[(q,r['bundle_sha256'])],r['best_seed_rrf_rank'],r['card_key'])); direct_struct_rank[(q,depth)]=[r['card_key'] for r in rows]
f2_bundle={(r['query_id'],r['card_key']):r for r in jl(P24/'followup2_bundles.jsonl') if r['cohort']=='and_combination'}; f5_bundle={(r['query_id'],int(r['depth']),r['card_key']):r for r in jl(P24/'followup5_bundles.jsonl')}; f5rank={(r['query_id'],int(r['depth'])):r for r in jl(P24/'followup5_rankings.jsonl')}
cmb_struct_rank={(q,20):f2_scored[q]['structural_card_keys'] for q in f2_scored}; cmb_struct_rank.update({(q,30):f5rank[(q,30)]['card_keys'] for q in f2_scored})
ranking_rows=[]
for q in sorted(query_text):
    ranking_rows.append({'query_id':q,'pipeline':'OLD','candidate_depth':20,'ranking_unit':'chunk','ordered_ids':old_ranked[q],'source':'21 frozen selective BGE or RRF; 24 F2 for AND'})
    for depth in (20,30): ranking_rows.append({'query_id':q,'pipeline':'STRUCT','candidate_depth':depth,'ranking_unit':'card','ordered_ids':direct_struct_rank[(q,depth)] if q in direct_q else cmb_struct_rank[(q,depth)],'source':'25 local bundle BGE' if q in direct_q else ('24 F2 bundle BGE' if depth==20 else '24 F5 bundle BGE')})
write_jsonl('ranking_freeze.jsonl',ranking_rows)
resources={'model':'bge-reranker-v2-m3','revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','local_files_only':True,'trust_remote_code':False,'dtype':'float16','batch_size':2,'max_length':8192,'truncation':'only_second','physical_gpu':0,'shared_ollama_allowed':True,'pair_count':len(scores),'finite_pairs':len(scores),'truncated_pairs':0,'load_seconds':load_seconds,'warmup_seconds':warmup_seconds,'scoring_seconds':scoring_seconds,'pairs_per_second':len(scores)/scoring_seconds,'peak_allocated_gib':peak_alloc,'peak_reserved_gib':peak_reserved,'gpu_before':gpu_before,'gpu_after':gpu_after,'bge_before':bge_before,'bge_after':tree(BGE),'external_api_network_new_embedding_chroma_package_install':0}; assert resources['bge_after']==bge_before; write_json('local_bge_resources.json',resources)
score_freeze={'status':'PASS_BEFORE_GOLD','pair_scores_sha256':sha(OUT/'local_bge_pair_scores.csv'),'audit_sha256':sha(OUT/'local_bge_scorer_audit.csv'),'ranking_sha256':sha(OUT/'ranking_freeze.jsonl'),'pair_count':len(scores),'finite':True,'gold_reads':0}; write_json('ranking_score_freeze.json',score_freeze)
assert {str(p.relative_to(ROOT)):sha(p) for p in INPUTS}==input_before and tree(BGE)==bge_before
print({'local_bge':'PASS','pairs':len(scores),'seconds':scoring_seconds,'oom':False,'truncated':0})


In [3]:
# CPU-only reconstruction from the exact score freeze created by the first local GPU attempt.
assert os.environ.get('RUN_APPROVED_25_LOCAL_BGE','0')=='0' and os.environ.get('RUN_APPROVED_25_EXTERNAL','0')=='0'
score_rows=list(csv.DictReader((OUT/'local_bge_pair_scores.csv').open(encoding='utf-8',newline=''))); audits=list(csv.DictReader((OUT/'local_bge_scorer_audit.csv').open(encoding='utf-8',newline=''))); resources=json.loads((OUT/'local_bge_resources.json').read_text()); score_freeze=json.loads((OUT/'ranking_score_freeze.json').read_text())
scores={(r['query_id'],r['bundle_sha256']):float(r['raw_logit']) for r in score_rows}; assert len(score_rows)==len(scores)==len(pair_plan)==327 and set(scores)==set(pair_plan) and all(math.isfinite(v) for v in scores.values())
assert len(audits)==327 and all(r['gold_or_evaluation_fields_used']=='False' and int(r['document_truncated_tokens'])==0 for r in audits) and resources['pair_count']==resources['finite_pairs']==327 and resources['truncated_pairs']==0 and resources['bge_before']==resources['bge_after']==bge_before
assert score_freeze['pair_scores_sha256']==sha(OUT/'local_bge_pair_scores.csv') and score_freeze['audit_sha256']==sha(OUT/'local_bge_scorer_audit.csv') and score_freeze['ranking_sha256']==sha(OUT/'ranking_freeze.jsonl')
for r in score_rows:
    assert r['query_text_sha256']==hashlib.sha256(query_text[r['query_id']].encode()).hexdigest() and r['card_key']==pair_plan[(r['query_id'],r['bundle_sha256'])]['card_key']
direct_struct_rank={}
for depth in (20,30):
    for q in direct_q:
        rows=[r for r in bundle_rows if r['query_id']==q and r['depth']==depth]; rows.sort(key=lambda r:(-scores[(q,r['bundle_sha256'])],r['best_seed_rrf_rank'],r['card_key'])); direct_struct_rank[(q,depth)]=[r['card_key'] for r in rows]
f2_bundle={(r['query_id'],r['card_key']):r for r in jl(P24/'followup2_bundles.jsonl') if r['cohort']=='and_combination'}; f5_bundle={(r['query_id'],int(r['depth']),r['card_key']):r for r in jl(P24/'followup5_bundles.jsonl')}; f5rank={(r['query_id'],int(r['depth'])):r for r in jl(P24/'followup5_rankings.jsonl')}
cmb_struct_rank={(q,20):f2_scored[q]['structural_card_keys'] for q in f2_scored}; cmb_struct_rank.update({(q,30):f5rank[(q,30)]['card_keys'] for q in f2_scored})
expected=[]
for q in sorted(query_text):
    expected.append({'query_id':q,'pipeline':'OLD','candidate_depth':20,'ranking_unit':'chunk','ordered_ids':old_ranked[q],'source':'21 frozen selective BGE or RRF; 24 F2 for AND'})
    for depth in (20,30): expected.append({'query_id':q,'pipeline':'STRUCT','candidate_depth':depth,'ranking_unit':'card','ordered_ids':direct_struct_rank[(q,depth)] if q in direct_q else cmb_struct_rank[(q,depth)],'source':'25 local bundle BGE' if q in direct_q else ('24 F2 bundle BGE' if depth==20 else '24 F5 bundle BGE')})
assert expected==jl(OUT/'ranking_freeze.jsonl') and tree(BGE)==bge_before
print({'local_bge_cache_restore':'PASS','pairs':327,'finite':327,'gpu_model_current_run':0,'external_requests':0})


{'local_bge_cache_restore': 'PASS', 'pairs': 327, 'finite': 327, 'gpu_model_current_run': 0, 'external_requests': 0}


In [4]:
# Payload construction remains before answer-gold semantic reads.
enc=tiktoken.get_encoding('cl100k_base'); old_card=lambda cid:old_by[cid]['metadata']['card_key']
def old_groups(q,k):
    ranked=old_ranked[q]; cards=[]
    for cid in ranked:
        card=old_card(cid)
        if card not in cards: cards.append(card)
        if len(cards)==k: break
    groups=[]
    for card in cards:
        ids=[cid for cid in ranked if old_card(cid)==card][:5]; first=old_by[ids[0]]['metadata']; groups.append({'card_key':card,'issuer':first['issuer'],'card_name':first['card_name'],'evidences':[{'source_id':cid,'heading':old_by[cid]['metadata'].get('section') or '', 'text':old_by[cid]['document'],'level':old_by[cid]['metadata']['level']} for cid in ids]})
    return groups
direct_bundle={(r['query_id'],int(r['depth']),r['card_key']):r for r in bundle_rows}
def struct_groups(q,depth,k):
    ranking=direct_struct_rank[(q,depth)] if q in direct_q else cmb_struct_rank[(q,depth)]; bundles=direct_bundle if q in direct_q else ({(a,b,c):v for (a,c),v in f2_bundle.items() for b in [20]} if depth==20 else f5_bundle)
    groups=[]
    for card in ranking[:k]:
        b=bundles[(q,depth,card)]; evidences=[]
        for i,cid in enumerate(b['selected_chunk_ids'][:5]):
            c=struct_by[cid]; heading=' > '.join(c['heading_path']) if c['heading_path'] else '(root content)'
            if i==0 and b.get('optional_parent_heading'): heading=b['optional_parent_heading']+' > '+heading
            evidences.append({'source_id':cid,'heading':heading,'text':c['evidence_text'],'level':'direct_body'})
        first=struct_by[b['selected_chunk_ids'][0]]['metadata']; groups.append({'card_key':card,'issuer':first['issuer'],'card_name':first['card_name'],'evidences':evidences,'bundle_sha256':b['bundle_sha256']})
    return groups
def sanitize(groups):
    sent=[]; audit=[]; evidence_number=1
    for gi,g in enumerate(groups,1):
        es=[]
        for e in g['evidences'][:5]:
            ids=enc.encode(e['text']); clipped=enc.decode(ids[:640]); eid=f'E{evidence_number:03d}'; evidence_number+=1; es.append({'evidence_id':eid,'heading':e['heading'],'text':clipped}); audit.append({'evidence_id':eid,'card_key':g['card_key'],'source_id':e['source_id'],'source_level':e['level'],'original_tokens':len(ids),'sent_tokens':len(enc.encode(clipped)),'truncated_tokens':max(0,len(ids)-640)})
        sent.append({'group_id':f'G{gi:02d}','issuer':g['issuer'],'card_name':g['card_name'],'evidences':es})
    return sent,audit
payloads=[]; payload_index=[]; contexts=[]; token_audit=[]
for config,cfg in CONFIGS.items():
    for order,q in enumerate(queries,1):
        qid=q['query_id']; k=cfg['input_unique_card_depth']; groups=old_groups(qid,k) if cfg['pipeline']=='OLD' else struct_groups(qid,cfg['candidate_depth'],k); sent,audit=sanitize(groups); assert len(sent)<=k and len({g['card_key'] for g in groups})==len(groups)
        evidence_json=json.dumps(sent,ensure_ascii=False,separators=(',',':')); user=USER_TEMPLATE.format(query=q['query_text'],evidence_json=evidence_json); request={'model':'gpt-5.6-terra','reasoning':{'effort':'medium'},'tools':[],'store':False,'max_output_tokens':1200,'input':[{'role':'system','content':[{'type':'input_text','text':SYSTEM_PROMPT}]},{'role':'user','content':[{'type':'input_text','text':user}]}],'text':{'format':{'type':'json_schema','name':'card_answer','strict':True,'schema':SCHEMAS[f'K{k}']}}}; pid=f'{config.lower()}-{qid}'; request_sha=hashlib.sha256(canon(request).encode()).hexdigest(); tokens=len(enc.encode(canon(request))); assert tokens<=24000
        assert request['text']['format']['schema']['properties']['cards']['maxItems']==k
        payloads.append({'payload_id':pid,'request_sha256':request_sha,'request':request}); payload_index.append({'payload_id':pid,'configuration':config,'query_id':qid,'cohort':q['cohort'],'task':q['task'],'input_unique_card_depth':k,'candidate_depth':cfg['candidate_depth'],'input_card_group_count':len(sent),'request_sha256':request_sha,'cl100k_serialized_request_tokens':tokens})
        contexts.append({'payload_id':pid,'configuration':config,'query_id':qid,'groups':groups,'sent_groups':sent,'evidence_audit':audit}); token_audit.append({'payload_id':pid,'configuration':config,'query_id':qid,'input_card_group_count':len(sent),'evidence_count':sum(len(g['evidences']) for g in sent),'cl100k_serialized_request_tokens':tokens,'truncated_evidence_count':sum(a['truncated_tokens']>0 for a in audit),'truncated_tokens':sum(a['truncated_tokens'] for a in audit)})
assert len(payloads)==len(payload_index)==len(contexts)==len(token_audit)==160 and len({r['payload_id'] for r in payloads})==160 and all(sum(r['configuration']==c for r in payload_index)==40 for c in CONFIGS)
allowed_group_keys=['group_id','issuer','card_name','evidences']; allowed_evidence_keys=['evidence_id','heading','text']
for p,c in zip(payloads,contexts):
    sent=c['sent_groups']; assert all(list(g)==allowed_group_keys and all(list(e)==allowed_evidence_keys for e in g['evidences']) for g in sent); assert p['request']['model']=='gpt-5.6-terra' and p['request']['reasoning']=={'effort':'medium'} and p['request']['tools']==[] and p['request']['store'] is False and p['request']['max_output_tokens']==1200
write_jsonl('contexts.jsonl',contexts); write_jsonl('payloads.jsonl',payloads); write_csv('payload_index.csv',payload_index); write_csv('payload_token_audit.csv',token_audit)
blind=[]; reveal=[]
for q in queries:
    for k in (3,5):
        old=f'old-k{k}-{q["query_id"]}'; struct=f'struct-k{k}-{q["query_id"]}'; flip=int(hashlib.sha256(f'{q["query_id"]}|{k}'.encode()).hexdigest(),16)%2; a,b=(old,struct) if flip==0 else (struct,old); pair_id='AB-'+hashlib.sha256(f'{q["query_id"]}|{k}'.encode()).hexdigest()[:12]; blind.append({'blind_pair_id':pair_id,'query_id':q['query_id'],'input_unique_card_depth':k,'blind_a_payload_id':a,'blind_b_payload_id':b}); reveal.append({'blind_pair_id':pair_id,'A_configuration':'OLD-K'+str(k) if flip==0 else 'STRUCT-K'+str(k),'B_configuration':'STRUCT-K'+str(k) if flip==0 else 'OLD-K'+str(k)})
assert len(blind)==len(reveal)==80; write_csv('manual_blind_pairs.csv',blind); write_csv('manual_blind_assignment_key.csv',reveal)
total_input=sum(int(r['cl100k_serialized_request_tokens']) for r in token_audit); output_cap=160*1200; input_cost=total_input*2/1_000_000; output_cost=output_cap*12/1_000_000
payload_manifest={'status':'FROZEN_BEFORE_ANSWER_GOLD','payload_rows':160,'configuration_query_cross_product':True,'request_cap':160,'single_run':True,'repeat_run_count':0,'model':'gpt-5.6-terra','reasoning_effort':'medium','tools':0,'store':False,'max_output_tokens_each':1200,'sdk_retries':0,'cl100k_method':'tokens in canonical serialized request JSON; exact under cl100k_base, not provider billing tokenizer','cost_semantics':'cl100k-based estimate upper bound; not a provider billing hard cap','total_input_tokens_cl100k':total_input,'max_input_tokens_per_request':max(int(r['cl100k_serialized_request_tokens']) for r in token_audit),'max_output_tokens_total':output_cap,'estimated_input_cost_usd_cl100k':input_cost,'estimated_output_cost_upper_bound_usd_at_token_cap':output_cost,'estimated_total_cost_upper_bound_usd_cl100k_not_provider_billing_cap':input_cost+output_cost,'payloads_sha256':sha(OUT/'payloads.jsonl'),'contexts_sha256':sha(OUT/'contexts.jsonl'),'payload_index_sha256':sha(OUT/'payload_index.csv'),'prompt_sha256':sha(OUT/'prompt.json'),'schema_sha256':sha(OUT/'response_schema.json'),'ranking_freeze_sha256':sha(OUT/'ranking_freeze.jsonl'),'external_requests_current':0}; write_json('payload_manifest.json',payload_manifest)
approval_core={'model':'gpt-5.6-terra','reasoning_effort':'medium','tools':0,'store':False,'strict_schema_sha256':sha(OUT/'response_schema.json'),'prompt_sha256':sha(OUT/'prompt.json'),'payloads_sha256':sha(OUT/'payloads.jsonl'),'ordered_request_sha256':[r['request_sha256'] for r in payloads],'item_cap':160,'request_cap':160,'input_token_cap_cl100k':total_input,'max_output_tokens_each':1200,'sdk_retries':0,'prohibited_transmission':['pipeline/configuration','K/depth','cohort/category','gold/cardinality','scores','paths/spans','labels','evaluation/gate fields']}; approval_core_sha=hashlib.sha256(canon(approval_core).encode()).hexdigest(); write_json('approval_core.json',{'approval_core':approval_core,'approval_core_sha256':approval_core_sha})
assert {str(p.relative_to(ROOT)):sha(p) for p in INPUTS}==input_before and tree(BGE)==bge_before
print({'payload_freeze':'PASS','payloads':160,'input_tokens_cl100k':total_input,'request_cap':160,'estimated_cost_upper_bound_usd_cl100k_not_provider_billing_cap':input_cost+output_cost,'external_requests':0})


{'payload_freeze': 'PASS', 'payloads': 160, 'input_tokens_cl100k': 1161780, 'request_cap': 160, 'estimated_cost_upper_bound_usd_cl100k_not_provider_billing_cap': 4.62756, 'external_requests': 0}


In [5]:
# Answer gold is first interpreted after ranking and payload hashes are frozen.
payload_manifest=json.loads((OUT/'payload_manifest.json').read_text()); assert payload_manifest['status']=='FROZEN_BEFORE_ANSWER_GOLD' and payload_manifest['payloads_sha256']==sha(OUT/'payloads.jsonl') and payload_manifest['ranking_freeze_sha256']==sha(OUT/'ranking_freeze.jsonl')
gold_rows=defaultdict(list)
for r in direct_source: gold_rows[r['query_id']].append(r)
direct_gold={}
for q,rows in gold_rows.items():
    vals={(r['expected_card'],r['expected_level'],r['required_terms']) for r in rows}; assert len(vals)==1; card,level,terms=next(iter(vals)); direct_gold[q]={'expected_card':card,'expected_level':level,'required_terms':json.loads(terms)}
card_ids={}
for r in csv.DictReader((P24/'gold_card_labels.csv').open(encoding='utf-8',newline='')): card_ids.setdefault(r['card_key'],r['card_id']); assert card_ids[r['card_key']]==r['card_id']
assert len(card_ids)==10
cmb_labels=[r for r in csv.DictReader((P24/'followup2_gold_labels.csv').open(encoding='utf-8',newline='')) if r['cohort']=='and_combination']; assert len(cmb_labels)==100
claim_by={r['claim_id']:r for r in jl(P24/'followup2_atomic_claim_audit.jsonl')}
unit_re=re.compile(r'(?<!\w)\d[\d,.]*\s*(?:원|%|퍼센트|포인트|마일|마일리지|회|개월|일|년|건|리터)(?!\w)')
condition_words=('전월','실적','한도','횟수','이상','이하','제외','조건','월 ','연간','건','회')
context_by={r['payload_id']:r for r in contexts}; index_by={r['payload_id']:r for r in payload_index}; answer_gold=[]; audits=[]; blockers=[]
for p in payloads:
    pid=p['payload_id']; idx=index_by[pid]; q=idx['query_id']; ctx=context_by[pid]; sent_evidence={e['evidence_id']:{'card_key':internal['card_key'],'text':e['text']} for internal,sent in zip(ctx['groups'],ctx['sent_groups']) for e in sent['evidences']}
    if idx['task']=='direct':
        g=direct_gold[q]; relevant=[{'card_id':card_ids[g['expected_card']],'card_key':g['expected_card'],'variant':'expected card'}]; source=[r for r in old_chunks if r['metadata']['card_key']==g['expected_card'] and all(norm(t) in norm(r['document']) for t in g['required_terms'])]; facts=[]
        for r in source:
            lines=[x.strip() for x in r['document'].splitlines() if x.strip() and any(norm(t) in norm(x) for t in g['required_terms'])]; facts.extend(lines or [r['document'].strip()])
        facts=list(dict.fromkeys(facts)); allowed_ids=[eid for eid,e in sent_evidence.items() if e['card_key']==g['expected_card'] and all(norm(t) in norm(e['text']) for t in g['required_terms'])]; conditions=[x for x in facts if any(w in x for w in condition_words)]; units=sorted(set(unit_re.findall(' '.join(facts)))); source_provenance=[{'chunk_id':r['id'],'card_key':r['metadata']['card_key'],'level':r['metadata']['level']} for r in source]
        row={'payload_id':pid,'query_id':q,'task':'direct','relevant_cards':relevant,'allowed_facts':facts,'allowed_units':units,'allowed_conditions':conditions,'conditions_not_applicable':not conditions,'allowed_evidence_ids':allowed_ids,'source_provenance':source_provenance,'abstain_rule':'If no allowed evidence ID supports the expected card and required terms, return insufficient evidence rather than inventing an answer.','recommendation_target':'not_applicable'}
    else:
        labels=[r for r in cmb_labels if r['query_id']==q and r['label']=='positive']; relevant=[{'card_id':r['card_id'],'card_key':r['card_key'],'variant':'condition-matching card'} for r in labels]; facts=[]; conditions=[]; source_provenance=[]; allowed_ids=[]
        for r in labels:
            claim_ids=json.loads(r['claim_ids_json']); roles=[role for cid in claim_ids for role in claim_by[cid]['roles']]; facts.extend(role['text'] for role in roles); conditions.extend(role['text'] for role in roles if role['role']=='condition'); source_provenance.extend({'claim_id':cid,'source_path':claim_by[cid]['source_path'],'source_sha256':claim_by[cid]['source_sha256'],'roles':claim_by[cid]['roles']} for cid in claim_ids)
            card_evidence={eid:e for eid,e in sent_evidence.items() if e['card_key']==r['card_key']}; combined='\n'.join(e['text'] for e in card_evidence.values())
            if all(norm(role['text']) in norm(combined) for role in roles):
                for eid,e in card_evidence.items():
                    if any(norm(role['text']) in norm(e['text']) for role in roles): allowed_ids.append(eid)
        facts=list(dict.fromkeys(facts)); conditions=list(dict.fromkeys(conditions)); units=sorted(set(unit_re.findall(' '.join(facts)))); allowed_ids=list(dict.fromkeys(allowed_ids)); row={'payload_id':pid,'query_id':q,'task':'recommendation','relevant_cards':relevant,'allowed_facts':facts,'allowed_units':units,'allowed_conditions':conditions,'conditions_not_applicable':not conditions,'allowed_evidence_ids':allowed_ids,'source_provenance':source_provenance,'abstain_rule':'Include only condition-matching cards whose required atomic claims are all supported by transmitted evidence; otherwise omit the card and mark insufficiency when nothing remains.','recommendation_target':'set_only; no subjective ordering gold'}
    complete=bool(row['task'] and row['relevant_cards'] and row['allowed_facts'] and row['source_provenance'] and row['abstain_rule'] and 'allowed_units' in row and 'allowed_conditions' in row and row['recommendation_target']); answer_gold.append(row); audits.append({'payload_id':pid,'query_id':q,'task':row['task'],'relevant_card_count':len(row['relevant_cards']),'allowed_fact_count':len(row['allowed_facts']),'allowed_unit_count':len(row['allowed_units']),'allowed_condition_count':len(row['allowed_conditions']),'allowed_evidence_id_count':len(row['allowed_evidence_ids']),'source_provenance_count':len(row['source_provenance']),'complete':complete,'abstain_expected':len(row['allowed_evidence_ids'])==0})
    if not complete: blockers.append({'payload_id':pid,'reason':'answer gold missing required task/card/fact/source/abstain field'})
assert len(answer_gold)==len(audits)==160 and len({r['payload_id'] for r in answer_gold})==160; write_jsonl('answer_gold.jsonl',answer_gold); write_csv('answer_gold_audit.csv',audits)
gold_freeze={'status':'PASS' if not blockers else 'BLOCKED','rows':160,'direct_rows':120,'recommendation_rows':40,'required_fields_complete':not blockers,'blockers':blockers,'ranking_sha256':sha(OUT/'ranking_freeze.jsonl'),'payload_sha256':sha(OUT/'payloads.jsonl'),'answer_gold_sha256':sha(OUT/'answer_gold.jsonl'),'audit_sha256':sha(OUT/'answer_gold_audit.csv'),'recommendation_is_condition_matching_set_only':True,'subjective_order_gold':False}; write_json('answer_gold_freeze.json',gold_freeze)
approval=json.loads((OUT/'approval_core.json').read_text()); status={'approval_core_sha256':approval['approval_core_sha256'],'status':'awaiting_explicit_external_api_approval' if not blockers else 'blocked_answer_gold_incomplete','external_execution_allowed':False,'external_requests_current':0,'blockers':blockers}; write_json('approval_status.json',status)
README='''# 25 — LLM 답변 품질 4조합 preflight

현재 10개 카드 문서 안의 개발 질의 40개로 OLD-K3/K5와 STRUCT-K3/K5 end-to-end 패키지를 비교하기 위한 입력을 동결했습니다. K는 입력으로 주는 서로 다른 카드 근거 그룹의 최대 수입니다. direct 답변은 필요한 카드만 가변적으로 내고, recommendation 답변은 최대 K개입니다. 같은 카드-group schema와 필드 순서, 카드당 근거 최대 5개, 근거당 cl100k_base 640-token head cap을 사용합니다. 실제 context 길이 차이는 허용하며 token-matched 비교는 하지 않습니다.

OLD/STRUCT 차이는 청킹 하나만이 아니라 후보 생성, query-text routing, BGE 적용 범위와 evidence packaging을 함께 포함합니다. 따라서 이후 결과도 청킹 단독 효과로 해석하면 안 됩니다. Proper/Numeric/Semantic/AND를 각각 보고하고 direct와 recommendation을 50:50으로 평균한 family macro를 중심으로 보며 전체 micro는 보조입니다.

외부 실행은 아직 승인되지 않았습니다. exact approval core가 별도로 승인되기 전까지 Responses API 요청은 0이며, schema/refusal/incomplete 응답은 retries 0 계약 아래 실패로 남깁니다. 이번 계획은 160개 single run뿐이고 반복 320회 실행은 포함하지 않습니다.

생성 설정은 `gpt-5.6-terra`, reasoning effort `medium`, tools 0, store=false, strict structured output, 응답당 최대 1,200 output tokens로 고정했습니다. K3/K5는 같은 schema 필드 구조를 쓰되 recommendation의 `cards.maxItems`를 각각 3/5로 고정합니다. 전송 승인 상한과 비용 추정은 `payload_manifest.json` 및 `approval_core.json`에 기록합니다. 비용은 cl100k_base 토큰 수와 출력 토큰 상한으로 계산한 estimate upper bound이며 provider billing hard cap이 아닙니다. 가격 URL은 이번 offline run에서 새로 조회하지 않았습니다.

STRUCT direct 질의용 로컬 BGE 점수는 327쌍을 한 번 생성했고 모두 finite, truncation/OOM 0이었습니다. 이후 CPU 단계의 join 오류를 고친 뒤에는 같은 score cache를 327/327 exact 검증해 재사용했으며 GPU 재점수는 하지 않았습니다. 최종 fresh-kernel 검증은 API/network/GPU/model/new embedding/Chroma 0입니다. 답변 gold 160행의 blocker가 없더라도 외부 Responses 실행은 별도 명시 승인 전까지 계속 차단합니다.
'''
(OUT/'README.md').write_text(README,encoding='utf-8')
execution_history={'attempts':[{'stage':'cpu_preflight','result':'PASS','external_api':0,'gpu':0},{'stage':'fresh_full_local_bge_then_downstream','local_bge_pairs_scored':327,'local_bge_result':'PASS','downstream_result':'FAILED_BEFORE_EXTERNAL_API','failure':'answer-gold internal/sanitized card-key join KeyError','external_api':0},{'stage':'fresh_cpu_saved_score_validation_and_finalize','result':'PASS' if not blockers else 'BLOCKED','saved_scores_reused':327,'gpu_model':0,'external_api':0}],'retry_policy':'local BGE was not rescored after downstream CPU failure'}; write_json('execution_history.json',execution_history)
output_names=['evaluation_contract.json','prompt.json','response_schema.json','queries.csv','input_manifest.json','local_preflight.json','structural_direct_bundles.jsonl','structural_direct_bundle_trace.csv','local_bge_pair_scores.csv','local_bge_scorer_audit.csv','local_bge_resources.json','ranking_score_freeze.json','ranking_freeze.jsonl','contexts.jsonl','payloads.jsonl','payload_index.csv','payload_token_audit.csv','manual_blind_pairs.csv','manual_blind_assignment_key.csv','payload_manifest.json','approval_core.json','answer_gold.jsonl','answer_gold_audit.csv','answer_gold_freeze.json','approval_status.json','execution_history.json','README.md']
input_after={str(p.relative_to(ROOT)):sha(p) for p in INPUTS}; bge_after=tree(BGE); assert input_after==input_before and bge_after==bge_before and payload_manifest['external_requests_current']==0
integrity={'status':'PASS' if not blockers else 'BLOCKED','compile_checked_separately':True,'execution_errors':0,'query_rows':40,'cohorts':{'proper_noun':10,'numeric_condition':10,'semantic':10,'and_recommendation':10},'payload_rows':160,'configuration_query_cross_product':True,'duplicate_payload_ids':0,'ranking_frozen_before_gold':True,'payload_frozen_before_gold':True,'answer_gold_rows':160,'answer_gold_complete':not blockers,'blind_pair_rows':80,'local_bge_pairs':len(scores),'local_bge_finite':True,'local_bge_truncated_pairs':0,'local_bge_scored_once_historical_attempt':True,'final_validation_saved_score_pairs':327,'final_validation_gpu_model':0,'payload_token_method':'canonical serialized request JSON under cl100k_base','external_api_requests':0,'network':0,'new_embedding':0,'chroma':0,'input_before':input_before,'input_after':input_after,'bge_before':bge_before,'bge_after':bge_after,'output_hashes':{n:sha(OUT/n) for n in output_names}}; write_json('integrity.json',integrity)
manifest={'phase':'offline_local_preflight','cpu_preflight_fresh_kernel':True,'local_bge_scoring_fresh_kernel_completed_before_downstream_join_failure':True,'final_cpu_saved_score_validation_fresh_kernel':True,'external_api_requests':0,'network':0,'new_embedding':0,'local_bge_pairs_scored_once':327,'local_bge_pairs_rescored_after_failure':0,'inputs':input_before,'outputs':{n:sha(OUT/n) for n in output_names},'self_hash_policy':'run_manifest.json and integrity.json excluded'}; write_json('run_manifest.json',manifest); integrity['run_manifest_sha256']=sha(OUT/'run_manifest.json'); write_json('integrity.json',integrity)
print({'preflight':'PASS' if not blockers else 'BLOCKED','payloads':160,'answer_gold_complete':not blockers,'approval_core_sha256':approval['approval_core_sha256'],'external_requests':0,'input_tokens_cl100k':payload_manifest['total_input_tokens_cl100k'],'estimated_cost_upper_bound_usd_cl100k_not_provider_billing_cap':payload_manifest['estimated_total_cost_upper_bound_usd_cl100k_not_provider_billing_cap']})


{'preflight': 'PASS', 'payloads': 160, 'answer_gold_complete': True, 'approval_core_sha256': 'ba2abf3c0e3b902b4849abf1fd1229523730cc8acbd1e04e4ec939b1849c2a74', 'external_requests': 0, 'input_tokens_cl100k': 1161780, 'estimated_cost_upper_bound_usd_cl100k_not_provider_billing_cap': 4.62756}


In [6]:
# Execute the frozen validator source only after payload and answer-gold freeze.
NOTEBOOK=ROOT/'notebooks/25_llm_answer_quality_pipeline_comparison.ipynb'
_notebook_doc=json.loads(NOTEBOOK.read_text())
_validator_cell=next(c for c in _notebook_doc['cells'] if c.get('id')=='25-response-validator-source')
assert _validator_cell.get('metadata',{}).get('tags')==['skip-execution']
exec(''.join(_validator_cell['source']))


{'response_validator': 'PASS', 'expected_abstention_rows': 11, 'synthetic_cases': 7, 'validator_code_sha256': '30f6b6180d6bf4da1d988ef55676a3f86b2ddd058d9281c594daf7afd5f108a5', 'execution_code_sha256': '458579c92d3a11f684bc1154411613d62037e09daa95988dcc1daef815148908', 'approval_core_sha256': '96cd9ca09cfde7ef573c83e7e58ed4da80ad8c1dc22f8f275c09cfc9a6d66007', 'execution_guard_sha256': '72f4cd2ff144e192b8525fe135bd5f177a869cbced2838f25fa3cba0859c0419', 'notebook_content_sha256': '743fe95666a8de5cb54af351b9250db0b4bc2a1a737f4978b0858f0a736f1fb0', 'external_requests': 0}


## External generation factual scoring

The frozen 160 responses are scored offline; transport validation and factual quality remain separate.


In [1]:
# Self-contained CPU/offline factual scoring of the frozen 160 responses.
from pathlib import Path
import csv,hashlib,json,os,re,unicodedata
ROOT=Path.cwd();ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT;OUT=ROOT/'notebooks/data/25_llm_answer_quality_pipeline_comparison';NB=ROOT/'notebooks/25_llm_answer_quality_pipeline_comparison.ipynb'
if os.environ.get('RUN_APPROVED_25_EXTERNAL','0')!='0':raise RuntimeError('external must be 0')
C=lambda x:json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'));H=lambda p:hashlib.sha256(p.read_bytes()).hexdigest();JL=lambda n:[json.loads(x) for x in (OUT/n).read_text().splitlines()]
def CR(n):
 with (OUT/n).open(encoding='utf-8',newline='') as f:return list(csv.DictReader(f))
def J(n,x):(OUT/n).write_text(json.dumps(x,ensure_ascii=False,indent=2,sort_keys=True)+'\n')
def W(n,x):(OUT/n).write_text(''.join(C(r)+'\n' for r in x))
def CSV(n,x,fields=None):
 x=list(x)
 with (OUT/n).open('w',encoding='utf-8',newline='') as f:w=csv.DictWriter(f,fieldnames=fields or list(x[0]));w.writeheader();w.writerows(x)
N=lambda x:' '.join(unicodedata.normalize('NFKC',str(x)).lower().split());T=lambda x:set(re.findall(r'[0-9a-z가-힣%]{2,}',N(x)));RX=re.compile(r'\d+(?:[.,]\d+)?\s*(?:%|만원|천원|원|개월|포인트|마일|회|일|년|점)');NUM=lambda x:{re.sub(r'[\s,]','',v) for v in RX.findall(N(x))}
def hit(a,b):
 aa,bb=T(a),T(b);return N(a) in N(b) or bool(aa) and NUM(a)<=NUM(b) and len(aa&bb)/len(aa)>=.6
def avg(x,k):return sum(float(r[k]) for r in x)/len(x)
contract={'declared_before_metrics':True,'cutoffs':[3,5],'quality_score':'mean(card_f1,citation_validity,grounded_claim_precision,required_fact_recall,numeric_condition_exact,complete_answer)','string_match':'NFKC lower; substring or .60 token recall; claim grounding .35 token recall; numeric exact subset','limitation':'No synonym/morphology model; paraphrase false negatives and lexical false positives are possible.','transport_failure_not_automatic_factual_error':True,'gate':'same-K STRUCT quality delta >=.025, wins>losses, all/cohort metrics non-regression; otherwise retain OLD','dev_single_run_no_promotion':True};J('answer_quality_scoring_contract.json',contract)
A='96cd9ca09cfde7ef573c83e7e58ed4da80ad8c1dc22f8f275c09cfc9a6d66007';G='72f4cd2ff144e192b8525fe135bd5f177a869cbced2838f25fa3cba0859c0419';ad=json.loads((OUT/'approval_core.json').read_text());gd=json.loads((OUT/'external_execution_guard.json').read_text())
if ad['approval_core_sha256']!=A or H(OUT/'external_execution_guard.json')=='' or gd['execution_guard_sha256']!=G:raise RuntimeError('approval')
rb,vb=H(OUT/'external_responses.jsonl'),H(OUT/'external_response_validation.jsonl');raw,val,gold,pay,ctx=map(JL,['external_responses.jsonl','external_response_validation.jsonl','answer_gold.jsonl','payloads.jsonl','contexts.jsonl']);idx,audit,blind=CR('payload_index.csv'),CR('answer_gold_audit.csv'),CR('manual_blind_pairs.csv')
sets=[raw,val,gold,pay,ctx,idx,audit]
if any(len(x)!=160 for x in sets):raise RuntimeError('rows')
ids=[[r['payload_id'] for r in x] for x in sets]
if any(x!=ids[0] for x in ids[1:]) or [x['request_sha256'] for x in pay]!=ad['approval_core']['ordered_request_sha256']:raise RuntimeError('keys')
R,V,Q,X,I,U=[{r['payload_id']:r for r in x} for x in [raw,val,gold,ctx,idx,audit]]
P={};usage=[];status={};refusals=0
for r in raw:
 texts=[];resp=r['response'];status[resp['status']]=status.get(resp['status'],0)+1
 for o in resp.get('output',[]):
  for p in o.get('content',[]):texts += [p.get('text','')] if p.get('type')=='output_text' else [];refusals+=p.get('type')=='refusal'
 try:P[r['payload_id']]=json.loads('\n'.join(texts)) if texts else None
 except:P[r['payload_id']]=None
 u=resp.get('usage') or {};usage.append((r['payload_id'],int(u.get('input_tokens',0)),int(u.get('output_tokens',0)),int(u.get('total_tokens',0))))
errs=sorted({e for r in val for e in r.get('quality_failures',[])+r.get('format_errors',[])});tax={'answer_type_task_mismatch':'task','card_identity_not_in_transmitted_group':'identity','claim_citation_crosses_card_group_ownership':'ownership','claim_has_no_minimum_lexical_grounding_in_owned_evidence':'grounding','expected_abstention_not_returned':'abstention'}
CSV('answer_quality_transport_taxonomy.csv',[{'error':e,'taxonomy':tax.get(e,'format_or_state'),'responses':sum(e in r.get('quality_failures',[])+r.get('format_errors',[]) for r in val),'not_automatic_wrong':True} for e in errs])
def S(pid,k):
 m,g,c,a=I[pid],Q[pid],X[pid],U[pid];z=P[pid];ok=isinstance(z,dict);expected=a['abstain_expected']=='True';identity={};owner={};ev={}
 for full,wire in zip(c['groups'],c['sent_groups']):
  q=(wire['issuer'],wire['card_name']);identity[q]=full['card_key'];owner[q]={e['evidence_id'] for e in wire['evidences']}
  for e in wire['evidences']:ev[e['evidence_id']]=e['text']
 cards=z.get('cards',[])[:k] if ok else [];names={x.get('card_name') for x in cards};claims=[x for x in z.get('claims',[]) if x.get('card_name') in names] if ok else [];keys=[identity.get((x.get('issuer'),x.get('card_name'))) for x in cards];refs=[];ground=[];text=[]
 for x in cards:refs+=x.get('citations',[]);text.append(str(x.get('summary','')))
 for x in claims:
  refs+=x.get('citations',[]);text.append(str(x.get('claim','')));matches=[q for q in identity if q[1]==x.get('card_name')];owned=owner.get(matches[0],set()) if len(matches)==1 else set();cite=' '.join(ev[e] for e in x.get('citations',[]) if e in ev);aa,bb=T(x.get('claim','')),T(cite);ground.append(bool(x.get('citations')) and all(e in owned for e in x.get('citations',[])) and NUM(x.get('claim',''))<=NUM(cite) and len(aa&bb)/max(1,len(aa))>=.35)
 pred={x for x in keys if x};rel={x['card_key'] for x in g['relevant_cards']};eff=set() if expected else rel;tp=len(pred&eff);prec=tp/len(pred) if pred else float(not eff);rec=tp/len(eff) if eff else float(not pred);f1=2*prec*rec/(prec+rec) if prec+rec else 0;cit=sum(e in ev for e in refs)/len(refs) if refs else float(expected);gp=sum(ground)/len(ground) if ground else float(expected);answer=' '.join(text);facts=list(dict.fromkeys(N(x) for x in g['allowed_facts']));fr=sum(hit(x,answer) for x in facts)/len(facts) if facts else 1;req=set().union(*(NUM(x) for x in g['allowed_units'])) if g['allowed_units'] else set();conds=g['allowed_conditions'];ne=float((not req or req<=NUM(answer)) and (g['conditions_not_applicable'] or not conds or all(hit(x,answer) for x in conds)));ins=ok and (z.get('answer_type')=='insufficient' or z.get('insufficient_evidence') is True);unsupported=len([x for x in keys if x is None])+len(pred-rel);falsea=int(expected and not ins);falsei=int(not expected and ins);complete=int(ok and R[pid]['response']['status']=='completed' and ins==expected and pred==eff and fr==cit==gp==ne==1 and not unsupported);critical=int(not ok or R[pid]['response']['status']!='completed' or falsea or unsupported);major=int(not critical and (falsei or rec<1 or fr<1 or cit<1 or gp<1 or ne<1));q=(f1+cit+gp+fr+ne+complete)/6
 return {'payload_id':pid,'configuration':m['configuration'],'native_k':int(m['input_unique_card_depth']),'evaluation_k':k,'query_id':m['query_id'],'cohort':m['cohort'],'task':m['task'],'transport_status':V[pid]['status'],'card_precision':prec,'card_recall':rec,'card_f1':f1,'citation_validity':cit,'grounded_claim_precision':gp,'required_fact_recall':fr,'numeric_value_unit_condition_exact':ne,'complete_answer':complete,'quality_score':q,'false_abstention':falsei,'false_answer':falsea,'unsupported_card_count':unsupported,'unsupported_claim_count':sum(not x for x in ground),'duplicate_card_count':len(keys)-len(set(keys)),'critical_error':critical,'major_error':major,'minor_error':int(not critical and not major and V[pid]['status']!='transport_semantic_validation_pass'),'error_severity':'critical' if critical else 'major' if major else 'minor' if V[pid]['status']!='transport_semantic_validation_pass' else 'none'}
scores=[S(p,k) for p in ids[0] for k in (3,5)];W('answer_quality_scores.jsonl',scores)
metrics=['card_precision','card_recall','card_f1','citation_validity','grounded_claim_precision','required_fact_recall','numeric_value_unit_condition_exact','complete_answer','quality_score','false_abstention','false_answer','unsupported_card_count','unsupported_claim_count','duplicate_card_count','critical_error','major_error','minor_error'];summary=[]
for conf in sorted({x['configuration'] for x in scores}):
 for k in (3,5):
  b=[x for x in scores if x['configuration']==conf and x['evaluation_k']==k]
  for scope,z in [('all',b),('direct',[x for x in b if x['task']=='direct']),('recommendation',[x for x in b if x['task']=='recommendation'])]+[(c,[x for x in b if x['cohort']==c]) for c in ['proper_noun','numeric_condition','semantic','and_recommendation']]:summary.append({'configuration':conf,'evaluation_k':k,'scope':scope,'query_count':len(z),**{m:avg(z,m) for m in metrics}})
CSV('answer_quality_summary.csv',summary);native=[x for x in summary if x['evaluation_k']==int(x['configuration'].split('K')[-1])];paired=[];wlt=[];reg=[]
for k in (3,5):
 o={x['query_id']:x for x in scores if x['configuration']==f'OLD-K{k}' and x['evaluation_k']==k};n={x['query_id']:x for x in scores if x['configuration']==f'STRUCT-K{k}' and x['evaluation_k']==k}
 for q in o:
  for m in ['quality_score','card_f1','complete_answer','required_fact_recall','grounded_claim_precision','citation_validity','critical_error']:
   d=n[q][m]-o[q][m];paired.append({'k':k,'query_id':q,'cohort':o[q]['cohort'],'metric':m,'old':o[q][m],'struct':n[q][m],'delta':d,'outcome':'win' if d>1e-12 else 'loss' if d<-1e-12 else 'tie'})
  if n[q]['quality_score']<o[q]['quality_score']-1e-12:reg.append({'k':k,'query_id':q,'cohort':o[q]['cohort'],'old':o[q]['quality_score'],'struct':n[q]['quality_score'],'delta':n[q]['quality_score']-o[q]['quality_score']})
 for scope in ['all','proper_noun','numeric_condition','semantic','and_recommendation']:
  z=[x for x in paired if x['k']==k and x['metric']=='quality_score' and (scope=='all' or x['cohort']==scope)];wlt.append({'k':k,'scope':scope,'wins':sum(x['outcome']=='win' for x in z),'losses':sum(x['outcome']=='loss' for x in z),'ties':sum(x['outcome']=='tie' for x in z),'n':len(z),'mean_delta':sum(x['delta'] for x in z)/len(z)})
CSV('answer_quality_paired.csv',paired);CSV('answer_quality_wlt.csv',wlt);CSV('answer_quality_regressions.csv',reg,['k','query_id','cohort','old','struct','delta'])
guards=['card_f1','citation_validity','grounded_claim_precision','required_fact_recall','numeric_value_unit_condition_exact','complete_answer'];cand=[]
for k in (3,5):
 checks=[]
 for scope in ['all','proper_noun','numeric_condition','semantic','and_recommendation']:
  o=next(x for x in native if x['configuration']==f'OLD-K{k}' and x['scope']==scope);n=next(x for x in native if x['configuration']==f'STRUCT-K{k}' and x['scope']==scope);checks += [n[m]>=o[m]-1e-12 for m in guards]+[n['critical_error']<=o['critical_error']+1e-12]
 w=next(x for x in wlt if x['k']==k and x['scope']=='all');o=next(x for x in native if x['configuration']==f'OLD-K{k}' and x['scope']=='all');n=next(x for x in native if x['configuration']==f'STRUCT-K{k}' and x['scope']=='all');d=n['quality_score']-o['quality_score'];cand.append({'k':k,'delta':d,'wins':w['wins'],'losses':w['losses'],'guardrails':all(checks),'eligible':all(checks) and d>=.025 and w['wins']>w['losses']})
decision={'selection':'select_struct_candidate' if any(x['eligible'] for x in cand) else 'retain_old_end_to_end_package','candidates':cand,'raw_best':max((x for x in native if x['scope']=='all'),key=lambda x:x['quality_score'])['configuration'],'regressions':len(reg),'dev_only_no_promotion':True};J('answer_quality_decision.json',decision)
it=sum(x[1] for x in usage);ot=sum(x[2] for x in usage);resources={'requests':160,'request_errors':0,'input_tokens':it,'output_tokens':ot,'total_tokens':sum(x[3] for x in usage),'cost_usd':it*2e-6+ot*12e-6,'provider_status':status,'refusals':refusals,'latency':'unmeasured','price':json.loads((OUT/'evaluation_contract.json').read_text())['price_contract']};J('answer_quality_resources.json',resources);J('answer_quality_summary.json',{'summary':summary,'decision':decision,'taxonomy':errs,'resources':resources})
CSV('answer_quality_blind_audit.csv',[{'blind_pair_id':x['blind_pair_id'],'query_id':x['query_id'],'k':x['input_unique_card_depth'],'a_response_sha256':hashlib.sha256(C(R[x['blind_a_payload_id']]).encode()).hexdigest(),'b_response_sha256':hashlib.sha256(C(R[x['blind_b_payload_id']]).encode()).hexdigest(),'human_preference':'','notes':''} for x in blind])
if H(OUT/'external_responses.jsonl')!=rb or H(OUT/'external_response_validation.jsonl')!=vb or len(scores)!=320:raise RuntimeError('raw/final')
names=['answer_quality_scoring_contract.json','answer_quality_scores.jsonl','answer_quality_summary.csv','answer_quality_summary.json','answer_quality_paired.csv','answer_quality_wlt.csv','answer_quality_transport_taxonomy.csv','answer_quality_regressions.csv','answer_quality_resources.json','answer_quality_decision.json','answer_quality_blind_audit.csv'];st=json.loads((OUT/'external_run_status.json').read_text());st.update({'factual_quality_scoring_complete':True,'status':'responses_and_cpu_factual_scoring_complete','score_rows':320,'raw_responses_sha256':rb,'response_validation_sha256':vb});J('external_run_status.json',st);names+=['external_run_status.json']
doc=json.loads(NB.read_text());stable={'nbformat':doc['nbformat'],'nbformat_minor':doc['nbformat_minor'],'cells':[{'cell_type':c['cell_type'],'id':c.get('id'),'source_text':''.join(c.get('source',[])) if isinstance(c.get('source',[]),list) else c.get('source',''),'tags':c.get('metadata',{}).get('tags',[])} for c in doc['cells']]};nh=hashlib.sha256(C(stable).encode()).hexdigest();man=json.loads((OUT/'run_manifest.json').read_text());outs=man.get('outputs',{})
for n in names:outs[n]=H(OUT/n)
outs['external_responses.jsonl']=rb;outs['external_response_validation.jsonl']=vb;man.update({'phase':'external_and_factual_scoring_complete','historical_api_requests':160,'current_scoring_api_network_gpu_model':0,'outputs':dict(sorted(outs.items())),'notebook_content_sha256':nh});J('run_manifest.json',man);J('answer_quality_integrity.json',{'status':'PASS','rows':[160,160,160,320],'raw_sha256':rb,'validation_sha256':vb,'raw_unchanged':H(OUT/'external_responses.jsonl')==rb,'current_api_network_gpu_model':0,'notebook_content_sha256':nh,'run_manifest_sha256':H(OUT/'run_manifest.json'),'output_hashes':man['outputs']})
print({'PASS':True,'input':it,'output':ot,'cost':resources['cost_usd'],'decision':decision['selection'],'raw_best':decision['raw_best'],'regressions':len(reg)})


{'PASS': True, 'input': 759735, 'output': 71419, 'cost': 2.3764979999999998, 'decision': 'retain_old_end_to_end_package', 'raw_best': 'STRUCT-K3', 'regressions': 20}


## Manual blind A/B audit packet

This CPU-only follow-up packages anonymous answers and their actually cited evidence. It does not change the automatic decision.


In [1]:
# CPU-only reviewer packet: anonymous same-K answers plus only their cited evidence.
from pathlib import Path
import csv,hashlib,json,os
ROOT=Path.cwd();ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT;OUT=ROOT/'notebooks/data/25_llm_answer_quality_pipeline_comparison';NB=ROOT/'notebooks/25_llm_answer_quality_pipeline_comparison.ipynb'
if os.environ.get('RUN_APPROVED_25_EXTERNAL','0')!='0' or os.environ.get('RUN_APPROVED_25_LOCAL_BGE','0')!='0':raise RuntimeError('offline guard')
canon=lambda x:json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'));sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
def jl(n):return [json.loads(x) for x in (OUT/n).read_text().splitlines()]
def cr(n):
 with (OUT/n).open(encoding='utf-8',newline='') as f:return list(csv.DictReader(f))
raw_before=sha(OUT/'external_responses.jsonl');key_before=sha(OUT/'manual_blind_assignment_key.csv');raw={x['payload_id']:x for x in jl('external_responses.jsonl')};ctx={x['payload_id']:x for x in jl('contexts.jsonl')};pairs=cr('manual_blind_pairs.csv');queries={x['query_id']:x for x in cr('queries.csv')}
def answer(pid):
 texts=[p.get('text','') for o in raw[pid]['response'].get('output',[]) if o.get('type')=='message' for p in o.get('content',[]) if p.get('type')=='output_text']
 try:return json.loads('\n'.join(texts))
 except:return {'answer_type':'response_failure','summary':'','cards':[],'claims':[],'citations':[],'insufficient_evidence':True}
def evidence(pid,obj):
 cited=set()
 for c in obj.get('cards',[]):cited.update(c.get('citations',[]))
 for c in obj.get('claims',[]):cited.update(c.get('citations',[]))
 cited.update(c.get('evidence_id') for c in obj.get('citations',[]))
 out=[]
 for g in ctx[pid]['sent_groups']:
  for e in g['evidences']:
   if e['evidence_id'] in cited:out.append({'evidence_id':e['evidence_id'],'issuer':g['issuer'],'card_name':g['card_name'],'heading':e['heading'],'text':e['text']})
 if {x['evidence_id'] for x in out}!={x for x in cited if x}:raise RuntimeError('cited evidence coverage')
 return out
rubric={'card_selection_accuracy':'','required_facts_numbers_conditions':'','citation_actual_support':'','omissions':'','unsupported_claims':'','response_failure':'','overall_A_B_or_tie':'','notes':''};rows=[]
for p in pairs:
 q=queries[p['query_id']];a=answer(p['blind_a_payload_id']);b=answer(p['blind_b_payload_id'])
 rows.append({'blind_pair_id':p['blind_pair_id'],'query_text':q['query_text'],'task':q['task'],'cohort':q['cohort'],'K':int(p['input_unique_card_depth']),'A':{'answer':a,'cited_evidence':evidence(p['blind_a_payload_id'],a)},'B':{'answer':b,'cited_evidence':evidence(p['blind_b_payload_id'],b)},'review_rubric':dict(rubric)})
forbidden={'configuration','pipeline','payload_id','old','struct','rank','score','path','gold','assignment'}
def keys(x):
 if isinstance(x,dict):
  for k,v in x.items():yield str(k).lower();yield from keys(v)
 elif isinstance(x,list):
  for v in x:yield from keys(v)
if len(rows)!=80 or len({x['blind_pair_id'] for x in rows})!=80 or any(k in forbidden for r in rows for k in keys(r)):raise RuntimeError('packet count/key leak')
text=''.join(canon(r)+'\n' for r in rows)
for token in ['"configuration"','"pipeline"','"payload_id"','"rank"','"score"','"path"','"gold"','"assignment"']:
 if token in text.lower():raise RuntimeError('assignment metadata leak '+token)
(OUT/'answer_quality_blind_packet.jsonl').write_text(text)
contract={'name':'25 same-K anonymous A/B manual audit packet','rows':80,'manual_followup_only':True,'automatic_decision_unchanged':True,'included':['query_text','task','cohort','K','anonymous A/B answer JSON','only actually cited evidence issuer/card/heading/text'],'forbidden':['configuration','pipeline','payload_id','OLD','STRUCT','rank','score','path','gold','assignment'],'rubric_fields':list(rubric),'external_api_network_gpu_model':0}
(OUT/'answer_quality_blind_packet_contract.json').write_text(json.dumps(contract,ensure_ascii=False,indent=2,sort_keys=True)+'\n')
integrity={'status':'PASS','rows':80,'unique_blind_pairs':80,'assignment_leak_string_count':0,'raw_response_sha256':raw_before,'raw_response_unchanged':sha(OUT/'external_responses.jsonl')==raw_before,'assignment_key_sha256':key_before,'assignment_key_unchanged':sha(OUT/'manual_blind_assignment_key.csv')==key_before,'packet_sha256':sha(OUT/'answer_quality_blind_packet.jsonl'),'contract_sha256':sha(OUT/'answer_quality_blind_packet_contract.json'),'external_api_network_gpu_model':0}
(OUT/'answer_quality_blind_packet_integrity.json').write_text(json.dumps(integrity,ensure_ascii=False,indent=2,sort_keys=True)+'\n')
man=json.loads((OUT/'run_manifest.json').read_text());man['outputs'].update({n:sha(OUT/n) for n in ['answer_quality_blind_packet.jsonl','answer_quality_blind_packet_contract.json','answer_quality_blind_packet_integrity.json']});man['blind_manual_audit_packet_rows']=80;man['current_blind_packet_api_network_gpu_model']=0;(OUT/'run_manifest.json').write_text(json.dumps(man,ensure_ascii=False,indent=2,sort_keys=True)+'\n')
aq=json.loads((OUT/'answer_quality_integrity.json').read_text());aq.update({'blind_packet_rows':80,'blind_packet_assignment_leaks':0,'blind_packet_sha256':sha(OUT/'answer_quality_blind_packet.jsonl'),'blind_packet_manual_followup_only':True,'run_manifest_sha256':sha(OUT/'run_manifest.json')});(OUT/'answer_quality_integrity.json').write_text(json.dumps(aq,ensure_ascii=False,indent=2,sort_keys=True)+'\n')
if sha(OUT/'external_responses.jsonl')!=raw_before or sha(OUT/'manual_blind_assignment_key.csv')!=key_before:raise RuntimeError('frozen source changed')
print({'blind_packet':'PASS','rows':80,'leaks':0,'packet_sha256':integrity['packet_sha256']})


{'blind_packet': 'PASS', 'rows': 80, 'leaks': 0, 'packet_sha256': '49ff3ae4298e6beb88c51b2bb335c3650dbf18daee8582f854b18747b725ea31'}


## Blind custom-agent review summary

This append-only follow-up records the post-judgment configuration decode without replacing the automatic decision.


In [1]:
# Append-only blind custom-agent review summary; no response rescoring or decision replacement.
from pathlib import Path
import hashlib,json,os
ROOT=Path.cwd();ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT;OUT=ROOT/'notebooks/data/25_llm_answer_quality_pipeline_comparison';NB=ROOT/'notebooks/25_llm_answer_quality_pipeline_comparison.ipynb'
if os.environ.get('RUN_APPROVED_25_EXTERNAL','0')!='0' or os.environ.get('RUN_APPROVED_25_LOCAL_BGE','0')!='0':raise RuntimeError('offline guard')
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest();raw_before=sha(OUT/'external_responses.jsonl');validation_before=sha(OUT/'external_response_validation.jsonl')
decision_path=OUT/'answer_quality_decision.json';decision=json.loads(decision_path.read_text())
if decision.get('selection')!='retain_old_end_to_end_package':raise RuntimeError('automatic decision changed')
summary={
 'provenance':{'reviewer':'single custom Codex reviewer','human_audit':False,'configuration_key_decoded_only_after_judgment':True,'review_input':'anonymous packet only','original_raw_outside_packet_evidence_read':False},
 'blind_pair_count':80,
 'decoded_results':{
  'overall':{'STRUCT':{'wins':7,'losses':11,'ties':62},'OLD':{'wins':11,'losses':7,'ties':62}},
  'K3':{'STRUCT':{'wins':4,'losses':6,'ties':30},'OLD':{'wins':6,'losses':4,'ties':30}},
  'K5':{'STRUCT':{'wins':3,'losses':5,'ties':32},'OLD':{'wins':5,'losses':3,'ties':32}},
  'direct':{'STRUCT':{'wins':1,'losses':6,'ties':53},'OLD':{'wins':6,'losses':1,'ties':53}},
  'recommendation':{'STRUCT':{'wins':6,'losses':5,'ties':9},'OLD':{'wins':5,'losses':6,'ties':9}},
  'breakdown':{'K3_direct':{'STRUCT':{'wins':1,'losses':3,'ties':26}},'K5_direct':{'STRUCT':{'wins':0,'losses':3,'ties':27}},'K3_recommendation':{'STRUCT':{'wins':3,'losses':3,'ties':4}},'K5_recommendation':{'STRUCT':{'wins':3,'losses':2,'ties':5}}}
 },
 'severity_with_common_errors_counted':{'STRUCT':{'critical':1,'major':11,'minor':5},'OLD':{'critical':0,'major':10,'minor':3}},
 'struct_critical_case':{'blind_pair_id':'AB-33bc686e4aed','error':'response_failure'},
 'automatic_decision_preserved':'retain_old_end_to_end_package',
 'final_interpretation':['old_accuracy_first_development_candidate','no_production_incumbent','switching_cost_not_applicable','not_eligible_for_promotion','dominance_not_established_62_of_80_ties'],
 'non_blind_automatic_regression_audit':{'provenance':'separate non-blind data analyst audit; not part of final blind result','audited_rows':20,'false_or_tie':13,'real_card_omission':2,'transport':3,'ambiguous':2}
}
(OUT/'answer_quality_blind_review_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,sort_keys=True)+'\n')
decision['blind_review_followup']={'automatic_selection_unchanged':True,'summary_sha256':sha(OUT/'answer_quality_blind_review_summary.json'),'final_interpretation':summary['final_interpretation'],'blind_overall':summary['decoded_results']['overall'],'dominance_not_established_ties':62,'custom_agent_not_human':True}
decision_path.write_text(json.dumps(decision,ensure_ascii=False,indent=2,sort_keys=True)+'\n')
readme=OUT/'README.md';marker='## Blind custom-agent review follow-up';text=readme.read_text();section='\n\n'+marker+'\n\nA single custom Codex reviewer judged only the anonymous packet, before configuration-key decoding; this was not a human audit. OLD led 11-7 with 62 ties overall, while STRUCT led recommendation 6-5 but lost direct 1-6. This preserves the automatic retain-OLD development decision. OLD is an accuracy-first development candidate, not a production incumbent; switching cost is not applicable, promotion is forbidden, and dominance is not established because 62/80 pairs tied. The separate 20-row non-blind regression audit is not merged into the blind result.\n';readme.write_text((text.split(marker)[0].rstrip() if marker in text else text.rstrip())+section)
manifest_path=OUT/'run_manifest.json';manifest=json.loads(manifest_path.read_text());manifest['outputs']['answer_quality_blind_review_summary.json']=sha(OUT/'answer_quality_blind_review_summary.json');manifest['outputs']['answer_quality_decision.json']=sha(decision_path);manifest['outputs']['README.md']=sha(readme);manifest['blind_custom_agent_review_rows']=80;manifest['blind_review_api_network_gpu_model']=0;manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2,sort_keys=True)+'\n')
integrity_path=OUT/'answer_quality_integrity.json';integrity=json.loads(integrity_path.read_text());integrity.update({'blind_review_summary_sha256':sha(OUT/'answer_quality_blind_review_summary.json'),'blind_review_rows':80,'blind_review_custom_agent_not_human':True,'automatic_decision_preserved':decision['selection']=='retain_old_end_to_end_package','raw_response_sha256_after_blind_review':sha(OUT/'external_responses.jsonl'),'validation_sha256_after_blind_review':sha(OUT/'external_response_validation.jsonl'),'blind_review_api_network_gpu_model':0,'run_manifest_sha256':sha(manifest_path)});integrity_path.write_text(json.dumps(integrity,ensure_ascii=False,indent=2,sort_keys=True)+'\n')
if sha(OUT/'external_responses.jsonl')!=raw_before or sha(OUT/'external_response_validation.jsonl')!=validation_before or decision['selection']!='retain_old_end_to_end_package':raise RuntimeError('frozen result changed')
print({'blind_review_followup':'PASS','OLD_WLT':[11,7,62],'STRUCT_WLT':[7,11,62],'automatic_decision':decision['selection']})


{'blind_review_followup': 'PASS', 'OLD_WLT': [11, 7, 62], 'STRUCT_WLT': [7, 11, 62], 'automatic_decision': 'retain_old_end_to_end_package'}


In [8]:
# Fail-closed external runner. This cell remains skipped until the exact execution guard is approved.
import os
def _require_external(condition,message):
    if not condition: raise RuntimeError(message)
_require_external(os.environ.get('RUN_APPROVED_25_EXTERNAL')=='1','External Responses execution is not approved')
_runner_doc=json.loads(NOTEBOOK.read_text())
def _runner_cell_sha(cell_id):
    cell=next(c for c in _runner_doc['cells'] if c.get('id')==cell_id); source=''.join(cell.get('source',[])) if isinstance(cell.get('source',[]),list) else cell.get('source',''); return hashlib.sha256(canon({'cell_id':cell_id,'source_text':source}).encode()).hexdigest()
current_validator_code_sha=_runner_cell_sha('25-response-validator-source'); current_execution_code_sha=_runner_cell_sha('25-external-fail-closed')
guard_doc=json.loads((OUT/'external_execution_guard.json').read_text()); guard=guard_doc['guard_core']; guard_sha=hashlib.sha256(canon(guard).encode()).hexdigest()
_require_external(guard_sha==guard_doc['execution_guard_sha256']==os.environ.get('APPROVED_25_EXECUTION_GUARD_SHA256',''),'Approved execution guard mismatch')
approval_doc=json.loads((OUT/'approval_core.json').read_text()); _require_external(hashlib.sha256(canon(approval_doc['approval_core']).encode()).hexdigest()==approval_doc['approval_core_sha256']==guard['approval_core_sha256'],'Approval core mismatch')
_require_external(current_validator_code_sha==guard['validator_code_sha256']==approval_doc['approval_core']['validator_code_sha256'] and current_execution_code_sha==guard['execution_code_sha256']==approval_doc['approval_core']['execution_code_sha256'],'Approved execution or validator code changed')
_require_external(sha(OUT/'payloads.jsonl')==guard['payloads_sha256'] and sha(OUT/'prompt.json')==guard['prompt_sha256'] and sha(OUT/'response_schema.json')==guard['schema_sha256'],'Payload, prompt, or schema hash mismatch')
_require_external(sha(OUT/'answer_gold.jsonl')==guard['answer_gold_sha256'] and sha(OUT/'answer_gold_freeze.json')==guard['answer_gold_freeze_sha256'] and sha(OUT/'response_validation_contract.json')==guard['response_validation_contract_sha256'],'Gold or validator contract hash mismatch')
_require_external(sha(OUT/'external_runner_optimized_guard_audit.json')==guard['optimized_guard_audit_sha256'],'Optimized guard audit hash mismatch')
_require_external(guard['response_count']==160 and guard['sdk_retries']==0 and len(payloads)==160,'Response count or retry contract mismatch')
from openai import OpenAI
client=OpenAI(max_retries=0); raw_path=OUT/'external_responses.jsonl'; validation_path=OUT/'external_response_validation.jsonl'; raw_path.write_text(''); validation_path.write_text('')
for item in payloads:
    pid=item['payload_id']
    try:
        response=client.responses.create(**item['request']); raw=response.model_dump(mode='json'); raw_record={'payload_id':pid,'request_sha256':item['request_sha256'],'response':raw}
    except Exception as exc:
        raw_record={'payload_id':pid,'request_sha256':item['request_sha256'],'request_error':type(exc).__name__}
        with raw_path.open('a',encoding='utf-8') as f: f.write(canon(raw_record)+'\n')
        result={'payload_id':pid,'status':'format_failure','format_errors':['request_error'],'quality_failures':[]}
        with validation_path.open('a',encoding='utf-8') as f: f.write(canon(result)+'\n')
        continue
    with raw_path.open('a',encoding='utf-8') as f: f.write(canon(raw_record)+'\n')
    try:
        texts=[part.get('text','') for out in raw.get('output',[]) if out.get('type')=='message' for part in out.get('content',[]) if part.get('type')=='output_text']; parsed=json.loads('\n'.join(texts)) if texts else None; result=validate_response(pid,parsed)
        if raw.get('status')!='completed' or not texts: result={'payload_id':pid,'status':'format_failure','format_errors':['provider_incomplete_refusal_or_no_output_json'],'quality_failures':[]}
    except Exception:
        result={'payload_id':pid,'status':'format_failure','format_errors':['response_json_parse_error'],'quality_failures':[]}
    with validation_path.open('a',encoding='utf-8') as f: f.write(canon(result)+'\n')
# Individual format/quality failures remain stored for scoring and do not abort the 160-response batch.
write_json('external_run_status.json',{'responses_saved':160,'transport_semantic_validation_saved':160,'factual_quality_scoring_complete':False,'required_post_execution_output':guard['required_post_execution_output'],'status':'responses_saved_awaiting_required_factual_quality_scoring'})
